<div align="center">

# Hafta 3: Derin öğrenme ve CNN temelleri

**Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı**<br>
Türk Kardiyoloji Derneği · Prof. Dr. Utku Köse

[Hafta sayfası](README.md) · [Ders sayfası](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html) · [Etkileşimli lab](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/lab.html) · [PDF ders notu](Hafta03_Ders_Notlari.pdf) · [Colab'da aç](https://colab.research.google.com/github/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-03/NB03_derin_ogrenme_ve_evrisimli_aglar.ipynb)

</div>

**Bu defter iki kısımdan oluşur.** Birincisinde kodu bir yapay zekâ (YZ) asistanı yazar; asistana ne yapacağını yazılı bir talimat, yani bir prompt (istem) anlatır. İkincisinde kodu Python ile kendiniz yazarsınız. İki kısım da aynı verilerle aynı sonuçlara varır; ders sayfasındaki her uygulama durağında bunlardan birini seçin.

| | Kimler için? | Ne yapılır? |
|---|---|---|
| **[Kısım 1: Prompt yolu](#part-1)** | Kod yazmayan katılımcılar | Kodu, hazır promptlarla bir YZ asistanı yazar; siz çalıştırır ve sonucu denetlersiniz. |
| **[Kısım 2: Python yolu](#part-2)** | Python öğrenmek isteyen katılımcılar | Kodu alıştırmalarla adım adım kendiniz yazar ve çalıştırırsınız. |

**Nerede çalışır?** Aynı defter, hiçbir kurulum gerektirmeyen Google Colab'da da kendi bilgisayarınızdaki Jupyter'de de (Anaconda; ders klasöründeki SETUP.tr.md dosyasına bakın) çalışır. Hangisini tercih ederseniz onu kullanın. Colab'da önce *Dosya → Drive'a kopya kaydet* (*File → Save a copy in Drive*) komutuyla kendi kopyanızı kaydedin; aksi hâlde sekme kapanınca yaptığınız değişiklikler kaybolur. Bu defterdeki ağların eğitimi sıradan bir dizüstü bilgisayarda hücre başına birkaç saniye ile yaklaşık bir dakika arasında sürer; ekran kartı gerekmez. Hangi kısmı seçerseniz seçin önce 0. bölümü çalıştırın. Bu bölüm ders dosyalarını ve paketleri hazırlar.

**Veri ve mahremiyet.** Buradaki bütün veri setleri ya sentetiktir ya da açık bir araştırma veri tabanından gelmektedir. Gerçek hasta verisi yalnızca kendi bilgisayarınızda, kurumunuzun kurallarına uygun olarak ve etik kurul onayıyla işlenir; bulut tabanlı bir deftere ya da bir YZ sohbetine asla konmaz.

## 0. Kurulum

İlk hücre ders klasörünü bulur. Defter tek başına açıldıysa, örneğin Google Colab'da, `vclinic` ders paketini ve veri dosyalarını GitHub'dan indirir. İkinci hücre paketleri yükler, mesajların dilini seçer ve haftanın isteğe bağlı paketlerini denetler. PyTorch, eğitimde kullanılan derin öğrenme paketidir. Colab'da hazır gelir; Anaconda için tek satırlık kurulum SETUP.tr.md dosyasında anlatılmaktadır. İkinci hücre ayrıca haftanın üç grubuna ait ACDC sine kardiyak manyetik rezonans dosyalarını (yaklaşık 105 MB) Human Heart Project'in (CREATIS, Lyon) açık veri tabanından indirir. İndirme bir iki dakika sürer; dosyalar zaten varsa bu adım atlanır. Veriler, ticari olmayan bilimsel araştırmalar için CC BY-NC-SA 4.0 lisansıyla ücretsiz olarak sunulmaktadır; her kullanımda Bernard ve arkadaşlarının IEEE Transactions on Medical Imaging dergisindeki 2018 tarihli makalesine atıf yapılmalıdır.

In [ ]:
#@title Ders dosyaları: Bu hücreyi çalıştırın (açmanıza gerek yok)
import pathlib
import sys
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/utkukose/ai-cardiology-tkd-NB-lecture/main/"
DATA_FILES = [
]


def find_course_root():
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:
        if (folder / "vclinic" / "__init__.py").exists():
            return folder
    return None


ROOT = find_course_root()
if ROOT is None or ROOT.name == "ai-cardiology-course":
    # İndirilmiş bir kopyada, örneğin Colab'da, paketin dosyaları yenilenir.
    ROOT = pathlib.Path.cwd() / "ai-cardiology-course" if ROOT is None else ROOT
    manifest = urllib.request.urlopen(REPO_RAW + "vclinic/MANIFEST.txt").read().decode("utf-8")
    package = ["vclinic/" + line.strip() for line in manifest.splitlines() if line.strip()]
    for rel in package + DATA_FILES:
        target = ROOT / rel
        if rel in package or not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(REPO_RAW + rel, target)
    print("Ders dosyaları şu klasöre indirildi:", ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
# Kurulum: paketler, dil ve çizim biçimi
import importlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import copy
import math

from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, roc_auc_score)
from vclinic import acdc, echonet
from vclinic.nifti import read_nifti

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ImportError as error:
    raise ImportError("PyTorch kurulu değil. SETUP.tr.md dosyasında anlatıldığı gibi kurun (pip install torch), ardından çekirdeği yeniden başlatın.") from error
import vclinic
from vclinic import teach

if tuple(int(x) for x in vclinic.__version__.split(".")) < (1, 4, 0):
    raise ImportError("vclinic ders paketi bu defterden daha eskidir. Ders klasörünü güncelleyin (git pull komutuyla ya da ZIP dosyasını yeniden indirerek) ve çekirdeği yeniden başlatın.")
teach.set_language("tr")
plt.rcParams.update({"figure.dpi": 110, "font.family": "DejaVu Sans"})
RANDOM_STATE = 2026
torch.manual_seed(RANDOM_STATE)
torch.set_num_threads(min(4, torch.get_num_threads()))
print("PyTorch", torch.__version__)
GROUPS = ("NOR", "DCM", "HCM")
acdc.download(GROUPS)
print("vclinic sürümü", vclinic.__version__)
print("Ders klasörü:", ROOT)
for name in ["ipywidgets", "torchvision", "cv2"]:
    try:
        importlib.import_module(name)
        print(f"{name}: kurulu")
    except ImportError:
        print(f"{name}: kurulu değil (isteğe bağlı; defter yerleşik bir yedek yol kullanır)")

In [ ]:
#@title Alıştırmaların beklenen cevapları (bu hücreyi çalıştırın; açmayı alıştırmaları denedikten sonraya bırakın)
teach.register_answers({
    "Alıştırma 3.1": (608256, 0),
    "Alıştırma 3.2": (104.5, 0.05),
    "Alıştırma 3.3": (0.323, 0.001),
    "Alıştırma A": (0.859, 0.001),
    "Alıştırma B": (-0.5, 0.001),
    "Alıştırma C": (90, 0),
    "Alıştırma D": (31, 0),
    "Alıştırma E": (12219, 0),
    "Alıştırma F": (1.429, 0.001),
    "Alıştırma G": (0.933, 0.001),
    "Alıştırma H": (4640, 0),
    "Alıştırma I": (73984, 0),
    "Alıştırma J": (0.933, 0.15),
    "Adım 1.1": (19.8, 0.05),
    "Adım 1.2": (62.2, 0.05),
    "Adım 1.3": (0.967, 0.001),
    "Adım 2.1": (0.2961, 0.0005),
    "Adım 2.2": (12219, 0),
    "Adım 2.3": (0.933, 0.15),
    "Adım 2.4": (0.933, 0.15),
})
print("Beklenen cevaplar hazır.")

In [ ]:
#@title Alıştırma promptlarının tabloları (bu hücreyi çalıştırın; Python yolunda okunması gerekmez)
def alistirma_tablosu(anahtar):
    """Bir alıştırma promptunun "veri" tablosu; Python yolunun tablolarından kurulur."""
    g = globals()
    k = anahtar
    if k == "3.1":
        v = g["vol"]
        return pd.DataFrame({"satır": [v.shape[0]], "sütun": [v.shape[1]], "kesit": [v.shape[2]]})
    if k == "3.2":
        return pd.DataFrame(g["img"])
    if k == "3.3":
        return pd.DataFrame({"LVEDVI": g["Xn"][:, 0], "en kalın duvar": g["Xn"][:, 1], "etiket": g["tn"].astype(int)})
    if k in ("C", "E"):
        names = {"train": "eğitim", "val": "doğrulama", "test": "test"}
        t = pd.DataFrame({"hasta": g["P"]["patient"], "küme": [names[s] for s in g["P"]["set"]]})
        if k == "E":
            t.insert(1, "grup", [g["P"]["classes"][c] for c in g["P"]["y"]])
            t["diyastol sonu"] = list(g["P"]["X"][:, 0])
            t["sistol sonu"] = list(g["P"]["X"][:, 1])
        return t
    if k == "J":
        net = g["course_model"]
        net.eval()
        with torch.no_grad():
            prob = F.softmax(net(g["Xte"]), dim=1).numpy()
        groups = ["NOR", "DCM", "HCM"]
        return pd.DataFrame({"hasta": g["P"]["patient"][g["te"]], "gerçek grup": [groups[c] for c in g["P"]["y"][g["te"]]],
                             "NOR": prob[:, 0], "DCM": prob[:, 1], "HCM": prob[:, 2]})
    raise KeyError(k)
print("Alıştırma promptlarının tabloları hazır.")

<a id="part-1"></a>

# Kısım 1: Prompt yolu

**Kimler için?** Bu kısım kod yazmayan katılımcılar içindir. Burada hiç kod yazmazsınız. Kodu, hazır bir prompttan, yani yazılı bir talimattan yararlanarak bir yapay zekâ (YZ) asistanı yazar; siz kodu çalıştırır ve sonucu denetlersiniz.

**Neye ihtiyacınız var?** Python kodu yazabilen bir YZ asistanı yeterlidir. Google Colab'ın içinde Gemini bulunur; pencerenin üstündeki Gemini düğmesiyle açılır. Başka bir tarayıcı sekmesinde açılan ChatGPT, Claude, Copilot ya da Gemini de aynı işi görür.

**Bir adım nasıl ilerler?**

1. Adımın amacını okuyun.
2. Promptun altındaki veri hücresini çalıştırın. Hücre, promptun tarif ettiği "veri" tablosunu hazırlar ve tablonun ilk ve son satırlarını gösterir.
3. Promptu kopyalayın: Fareyle seçin ve Ctrl+C'ye (Mac'te Cmd+C) basın.
4. Promptu YZ asistanına yapıştırın ve gönderin.
5. Asistanın verdiği kodu kopyalayın, veri hücresinin altındaki hücreye `#` ile başlayan satırın yerine yapıştırın ve hücreyi ▶ düğmesiyle ya da Shift+Enter ile çalıştırın.
6. Hemen altındaki kontrol hücresini çalıştırın. Hücre *doğru ✓* ya da *henüz değil ✗* yazar.
7. *Henüz değil* yazarsa aşağıdaki onarım promptunu kodla birlikte asistana gönderin. Sonuç yine yanlışsa *Hazır kod* hücresini çalıştırın. Bu hücre her zaman doğru sonucu verir; böylece bir sonraki adıma geçebilirsiniz.

**"veri" ve "cevap".** 1. haftada olduğu gibi her adımın veri hücresi, promptun tarif ettiği "veri" tablosunu hazırlar. Asistan sonucu "cevap" adıyla saklar; kontrol hücresi de sonucu bu addan okur.

**Her durağın başında** o durağın hazırlık hücresini bir kez çalıştırın. Hücre durağın verilerini yükler; böylece bir durağa başka bir gün de başlanabilir.

**Onarım promptu** (çalışan ama kontrolden geçmeyen kod için):

> Aşağıdaki kod şu görev için yazıldı: [promptun Görev satırını yapıştırın]. Kod çalışıyor, ancak dersin kontrolü "henüz değil" yazıyor. Hatayı bul, iki cümleyle açıkla ve düzeltilmiş kodu ver. Sonuç yine "cevap" adıyla saklanmalı.  
>
> [kodu buraya yapıştırın]  

**Hata mesajı için prompt** (hata vererek duran kod için):

> Aşağıdaki kod bir Jupyter defterinde çalıştırıldığında altta verilen hata mesajını veriyor. Hatanın nedenini sade bir dille açıkla ve düzeltilmiş kodu ver.  
>
> [kodu buraya yapıştırın]  
>
> [hata mesajını buraya yapıştırın]  

Haftanın bütün promptları [prompt kütüphanesinde](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-03/PROMPTLAR.md) de yer alır; GitHub her promptun yanında bir kopyalama düğmesi gösterir.

**Hiç değişmeyen kural.** Promptlarda yalnızca dersin sentetik ya da açık verileri bulunur. Bir YZ asistanına gerçek hasta verisi asla yapıştırılmaz.

## Durak 1: Sayılarla sine MR

*Ders: [1. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#stop-1)* · *Python yolunda aynı sonuç: Kısım 2, 1-3. bölümler*

ACDC'nin uzman konturları her incelemeyi sayılara dönüştürür: Hacimler, ejeksiyon fraksiyonu (EF) ve duvarların kalınlığı bu konturlardan ölçülür. Bu durakta üç grup bu sayılarla tanımlanır, 134 numaralı hastanın EF'si konturların vokselleriyle hesaplanır ve ACDC kuralları her hastaya uygulanır.

**Durağın sonunda:** DCM grubunun ortalama EF'sini (yüzde 19,8), 134 numaralı hastanın EF'sini (yüzde 62,2) ve ACDC kurallarının 90 hastadaki doğruluğunu (0,967) elde edeceksiniz.

In [ ]:
#@title 1. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
GROUPS = ("NOR", "DCM", "HCM")
acdc_table = acdc.table(GROUPS)
case_dir = acdc.default_folder() / "patient134"
info = acdc.read_info(case_dir / "Info.cfg")
gt_ed, spacing = read_nifti(case_dir / f"patient134_frame{int(info['ED']):02d}_gt.nii.gz")
gt_es, _ = read_nifti(case_dir / f"patient134_frame{int(info['ES']):02d}_gt.nii.gz")
print(acdc_table.shape, gt_ed.shape, spacing)
print("1. durak hazır.")

### Adım 1.1: Üç grubun ejeksiyon fraksiyonu

**Amaç.** Bu adımda EF'nin ACDC'nin üç grubunu nasıl ayırdığı görülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Kardiyak MR veri seti ACDC'de, sol ventrikül ejeksiyon fraksiyonunun normal kalpler, dilate kardiyomiyopati ve hipertrofik kardiyomiyopati arasında nasıl farklılaştığını görmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; ACDC'nin NOR (normal), DCM (dilate kardiyomiyopati) ve HCM (hipertrofik kardiyomiyopati) gruplarındaki 90 hastayı tutar. Her satır bir hastadır: "hasta" sütunu hastanın adını, "grup" grubunu, "LVEF" sol ventrikül ejeksiyon fraksiyonunu yüzde olarak, "LVEDVI" vücut yüzey alanına göre indekslenmiş sol ventrikül diyastol sonu hacmini mL/m² olarak, "en kalın duvar" da diyastol sonunda duvarın en kalın segmentini mm olarak verir. Bütün değerler uzmanların çizdiği konturlarda ölçülmüştür.  
> Görev: Her grubun ortalama LVEF değerini hesapla ve üç ortalamayı bir ondalık basamakla yazdır.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: DCM grubunun ortalama LVEF değerini bir ondalık basamakla "cevap" adıyla sakla.  
> Test: DCM grubunun ortalaması yüzde 40'ın altında, HCM grubununki yüzde 50'nin üzerinde olmalı.  

In [ ]:
#@title Adım 1.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = acdc_table[["patient", "group", "lv_ef", "lv_edvi", "max_wall_mm"]].rename(columns={
    "patient": "hasta", "group": "grup", "lv_ef": "LVEF", "lv_edvi": "LVEDVI", "max_wall_mm": "en kalın duvar"})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.1 kontrolü
teach.check_step("Adım 1.1", "cevap", globals())

In [ ]:
#@title Adım 1.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = acdc_table[["patient", "group", "lv_ef", "lv_edvi", "max_wall_mm"]].rename(columns={
    "patient": "hasta", "group": "grup", "lv_ef": "LVEF", "lv_edvi": "LVEDVI", "max_wall_mm": "en kalın duvar"})
ortalamalar = veri.groupby("grup")["LVEF"].mean().round(1)
print(ortalamalar)
cevap = float(ortalamalar["DCM"])
print("DCM'nin ortalama EF'si:", cevap)
teach.check_step("Adım 1.1", "cevap", globals())

**Sonuç ne gösteriyor?** DCM grubunun ortalama EF'si yüzde 19,8, normal grubunki yüzde 62,4, HCM grubununki ise yüzde 69,8'dir. ACDC'de düşük EF, DCM tanımının bir parçasıdır; HCM'de ise EF normal ya da yüksektir ve grupları duvar kalınlığı ayırır.

### Adım 1.2: 134 numaralı hastanın voksellerden hesaplanan ejeksiyon fraksiyonu

**Amaç.** Bu adımda, bir KMR laboratuvarının yazılımında olduğu gibi, hacimler ve EF uzman konturlarından kesit kesit hesaplanır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: ACDC'deki 134 numaralı hastanın sol ventrikül diyastol sonu ve sistol sonu hacimlerini ve ejeksiyon fraksiyonunu, konturla çizilmiş kaviteyi kesit kesit toplayarak hesaplamak istiyorum.  
> Veri: Defterde "veri" adında bir tablo var. Her satır, incelemenin tabandan apekse doğru sıralanmış bir kısa eksen kesitidir: "kesit" sütunu kesitin numarasını, "diyastol sonu" ve "sistol sonu" sütunları da o kesitte sol ventrikül kavitesinin konturu içinde kalan piksel sayısını verir. Bir piksel 1,445 × 1,445 mm'dir ve kesitler 10 mm kalınlığındadır.  
> Görev: Her faz için bütün kesitlerdeki pikselleri topla ve toplamı bir vokselin hacmiyle, yani 1,445 × 1,445 × 10 mm³ ile çarp; mL'ye çevirmek için 1.000'e böl. Böylece diyastol sonu hacmi EDV ve sistol sonu hacmi ESV bulunur. EF, (EDV − ESV)/EDV × 100 formülüyle hesaplanır.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: EDV ve ESV değerlerini mL, EF'yi yüzde olarak bir ondalık basamakla yazdır; EF'yi "cevap" adıyla sakla.  
> Test: EDV, ESV'den büyük olmalı ve 50 ile 300 mL arasında bulunmalı.  

In [ ]:
#@title Adım 1.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = pd.DataFrame({"kesit": np.arange(1, gt_ed.shape[2] + 1),
                     "diyastol sonu": [int((gt_ed[:, :, k] == 3).sum()) for k in range(gt_ed.shape[2])],
                     "sistol sonu": [int((gt_es[:, :, k] == 3).sum()) for k in range(gt_es.shape[2])]})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.2 kontrolü
teach.check_step("Adım 1.2", "cevap", globals())

In [ ]:
#@title Adım 1.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = pd.DataFrame({"kesit": np.arange(1, gt_ed.shape[2] + 1),
                     "diyastol sonu": [int((gt_ed[:, :, k] == 3).sum()) for k in range(gt_ed.shape[2])],
                     "sistol sonu": [int((gt_es[:, :, k] == 3).sum()) for k in range(gt_es.shape[2])]})
voksel_ml = 1.445 * 1.445 * 10 / 1000
edv = veri["diyastol sonu"].sum() * voksel_ml
esv = veri["sistol sonu"].sum() * voksel_ml
cevap = round(100 * (edv - esv) / edv, 1)
print("EDV:", round(edv, 1), "mL · ESV:", round(esv, 1), "mL · EF:", cevap, "%")
teach.check_step("Adım 1.2", "cevap", globals())

**Sonuç ne gösteriyor?** Diyastol sonundaki 5.319 ve sistol sonundaki 2.010 voksel, yüzde 62,2'lik bir EF verir. Konturlar kavitenin her vokselini kesit kesit sayar; 4. hafta aynı hacmi ekokardiyogramlarda disk yöntemiyle kestirmektedir.

### Adım 1.3: ACDC kurallarının her hastaya uygulanması

**Amaç.** Bu adımda 90 hasta ACDC tanımlarıyla sınıflanır ve ölçümlerin tek başına nerede yanıldığı görülür.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: ACDC tanımlarındaki ölçüm kurallarının bir hastayı ne sıklıkla uzmanların verdiği gruba yerleştirdiğini öğrenmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; ACDC'nin NOR (normal), DCM (dilate kardiyomiyopati) ve HCM (hipertrofik kardiyomiyopati) gruplarındaki 90 hastayı tutar. Her satır bir hastadır: "hasta" sütunu hastanın adını, "grup" grubunu, "LVEF" sol ventrikül ejeksiyon fraksiyonunu yüzde olarak, "LVEDVI" vücut yüzey alanına göre indekslenmiş sol ventrikül diyastol sonu hacmini mL/m² olarak, "en kalın duvar" da diyastol sonunda duvarın en kalın segmentini mm olarak verir. Bütün değerler uzmanların çizdiği konturlarda ölçülmüştür.  
> Görev: ACDC kurallarını her hastaya uygula: LVEF 40'ın altında ve LVEDVI 100'ün üzerindeyse hasta DCM'dir; değilse en kalın duvar 15'in üzerinde ve LVEF 50 ya da daha yüksekse HCM'dir; ikisi de değilse NOR'dur. Hastaları gruba ve kurala göre sayan bir çapraz tablo yazdır ve kuralı grubuyla aynı olan hastaların oranını hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Oranı üç ondalık basamakla yazdır ve "cevap" adıyla sakla.  
> Test: Çapraz tablodaki hastaların toplamı 90 olmalı ve oran 0 ile 1 arasında bulunmalı.  

In [ ]:
#@title Adım 1.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = acdc_table[["patient", "group", "lv_ef", "lv_edvi", "max_wall_mm"]].rename(columns={
    "patient": "hasta", "group": "grup", "lv_ef": "LVEF", "lv_edvi": "LVEDVI", "max_wall_mm": "en kalın duvar"})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 1.3 kontrolü
teach.check_step("Adım 1.3", "cevap", globals())

In [ ]:
#@title Adım 1.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
veri = acdc_table[["patient", "group", "lv_ef", "lv_edvi", "max_wall_mm"]].rename(columns={
    "patient": "hasta", "group": "grup", "lv_ef": "LVEF", "lv_edvi": "LVEDVI", "max_wall_mm": "en kalın duvar"})


def acdc_kurali(satir):
    if satir["LVEF"] < 40 and satir["LVEDVI"] > 100:
        return "DCM"
    if satir["en kalın duvar"] > 15 and satir["LVEF"] >= 50:
        return "HCM"
    return "NOR"


veri["kural"] = veri.apply(acdc_kurali, axis=1)
print(pd.crosstab(veri["grup"], veri["kural"]))
cevap = round(float((veri["kural"] == veri["grup"]).mean()), 3)
print("Kuralların doğruluğu:", cevap)
teach.check_step("Adım 1.3", "cevap", globals())

**Sonuç ne gösteriyor?** Kurallar hastaların yüzde 97 kadarını kendi grubuna yerleştirir. Üç hatanın üçü de sınırda kalan vakalardır: LVEDVI değeri 96,7 mL/m² olan bir DCM hastası, EF'si yüzde 41,5 olan bir DCM hastası ve uzman konturlarında en kalın segmenti 14,6 mm ölçülen bir HCM hastası. Uzmanlar bu üç sayıdan fazlasını değerlendirmiştir.

**1. durak tamamlandı.** [Ders sayfasına](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#stop-1) dönün ve duraktan sonrasını okumaya devam edin.

## Durak 2: İlk evrişimli ağ

*Ders: [2. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#stop-2)* · *Python yolunda aynı sonuç: Kısım 2, 4-7. bölümler*

Dersin ağı üç grubu ortadaki kesitlerin piksellerinden öğrenir; her kesitin diyastol sonu ve sistol sonu kareleri ağa iki kanal olarak girer. Bu durakta kesitler normalleştirilir, ağ kurulur ve erken durdurmayla eğitilir; ardından ağ 30 test hastasında, her hasta için tek bir kararla değerlendirilir.

**Durağın sonunda:** 12.219 parametreli bir ağ kuracak, ağın 30 test hastasındaki doğruluğunu (yaklaşık 0,933) ve makro F1 değerini (yaklaşık 0,933) bulacaksınız. Eğitim yaklaşık bir dakika sürer.

In [ ]:
#@title 2. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
GROUPS = ("NOR", "DCM", "HCM")
P = acdc.pairs(GROUPS)

class CmrCNN(nn.Module):
    """Üç evrişim bloğu ve bir doğrusal katman; girdi: bir kesitin iki karesi, iki kanal olarak."""

    def __init__(self, n_classes=3, dropout=0.3, batchnorm=True, in_channels=2):
        super().__init__()
        layers, c_in = [], in_channels
        for c_out in (8, 16, 32):
            layers += [nn.Conv2d(c_in, c_out, 3, padding=1)]
            layers += [nn.BatchNorm2d(c_out)] if batchnorm else []
            layers += [nn.ReLU(), nn.MaxPool2d(2)]
            c_in = c_out
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(32 * 8 * 8, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


def train_model(model, X_tr, y_tr, X_va, y_va, epochs=40, lr=0.003, batch=16, seed=RANDOM_STATE):
    torch.manual_seed(seed)
    order_gen = torch.Generator().manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    best, best_state, history = float("inf"), None, []
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(y_tr), generator=order_gen)
        total = 0.0
        for start in range(0, len(y_tr), batch):
            j = order[start:start + batch]
            loss = F.cross_entropy(model(X_tr[j]), y_tr[j])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(j)
        model.eval()
        with torch.no_grad():
            val_loss = F.cross_entropy(model(X_va), y_va).item()
        history.append((total / len(y_tr), val_loss))
        if val_loss < best:
            best, best_state = val_loss, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return history



def _course_decisions():
    """Dersin ağının 30 test hastası için kararları (bir kez eğitilir, yaklaşık bir dakika)."""
    if "_course_result" in globals():
        return _course_result
    X, y, sets, patients = P["X"], P["y"], P["set"], P["patient"]
    tr, va, te = sets == "train", sets == "val", sets == "test"
    m, sd = float(X[tr].mean()), float(X[tr].std())
    T = lambda a: torch.tensor((a - m) / sd, dtype=torch.float32)
    torch.manual_seed(RANDOM_STATE)
    net = CmrCNN()
    train_model(net, T(X[tr]), torch.tensor(y[tr]), T(X[va]), torch.tensor(y[va]))
    net.eval()
    with torch.no_grad():
        prob = F.softmax(net(T(X[te])), dim=1).numpy()
    names = sorted(set(patients[te]))
    mean_prob = np.array([prob[patients[te] == n].mean(axis=0) for n in names])
    true = np.array([y[te][patients[te] == n][0] for n in names])
    globals()["_course_result"] = {"names": names, "true": true, "pred": mean_prob.argmax(axis=1), "slice_prob": prob}
    return _course_result
print("2. durak hazır.")

### Adım 2.1: Eğitim kesitleriyle normalleştirme

**Amaç.** Bu adımda ağın girdileri, yalnızca eğitim hastalarından hesaplanan istatistiklerle hazırlanır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Bir ağ ACDC gruplarını sine MR kesitlerinden öğrenmeden önce, pikselleri yalnızca eğitim hastalarından gelen sayılarla ölçeklemek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; ACDC'deki 90 hastanın her birinin ortadaki üç kısa eksen kesitinden oluşan 270 sine MR kesitini tutar. Her satır bir kesittir: "hasta" sütunu hastayı, "grup" hastanın grubunu ("NOR", "DCM" ya da "HCM"), "küme" ise hastanın kümesini verir ("eğitim", 45 hasta; "doğrulama", 15 hasta; "test", 30 hasta). "diyastol sonu" ve "sistol sonu" sütunlarının her hücresinde, değerleri 0 ile 1 arasında olan 64 × 64 piksellik bir görüntü vardır; bir piksel 2 mm'dir.  
> Görev: Eğitim kesitlerinin, yani kümesi "eğitim" olan kesitlerin iki karesindeki bütün piksellerin ortalamasını ve standart sapmasını hesapla. Bütün kesitlerin piksellerini bu iki sayıyla normalleştir: Ortalamayı çıkar ve standart sapmaya böl.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Ortalamayı ve standart sapmayı dört ondalık basamakla yazdır; ortalamayı "cevap" adıyla sakla.  
> Test: Normalleştirmeden sonra eğitim piksellerinin ortalaması yaklaşık 0, standart sapması yaklaşık 1 olmalı; 135 eğitim kesiti 64 × 64 piksellik boyutunu korumalı.  

In [ ]:
#@title Adım 2.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_SETS = {"train": "eğitim", "val": "doğrulama", "test": "test"}
veri = pd.DataFrame({"hasta": P["patient"], "grup": [P["classes"][k] for k in P["y"]],
                     "küme": [_SETS[s] for s in P["set"]],
                     "diyastol sonu": list(P["X"][:, 0]), "sistol sonu": list(P["X"][:, 1])})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.1 kontrolü
teach.check_step("Adım 2.1", "cevap", globals())

In [ ]:
#@title Adım 2.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_SETS = {"train": "eğitim", "val": "doğrulama", "test": "test"}
veri = pd.DataFrame({"hasta": P["patient"], "grup": [P["classes"][k] for k in P["y"]],
                     "küme": [_SETS[s] for s in P["set"]],
                     "diyastol sonu": list(P["X"][:, 0]), "sistol sonu": list(P["X"][:, 1])})
goruntuler = np.stack([np.stack(veri["diyastol sonu"]), np.stack(veri["sistol sonu"])], axis=1)
egitim = (veri["küme"] == "eğitim").to_numpy()
mean, sd = float(goruntuler[egitim].mean()), float(goruntuler[egitim].std())
normal = (goruntuler - mean) / sd
print(normal[egitim].shape, "ortalama", round(mean, 4), "SS", round(sd, 4))
cevap = round(mean, 4)
teach.check_step("Adım 2.1", "cevap", globals())

**Sonuç ne gösteriyor?** Eğitim piksellerinin ortalaması 0,2961 olarak bulunur. Doğrulama ve test kesitleri de aynı iki sayıyla normalleştirilir; böylece onlardan eğitime hiçbir bilgi sızmaz.

### Adım 2.2: Ağ ve parametreleri

**Amaç.** Bu adımda dersin ağı kurulur ve ağırlıkları sayılır.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: ACDC grupları için küçük bir evrişimli ağ kurmak ve ağın kaç ağırlık öğrendiğini bilmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; ACDC'deki 90 hastanın her birinin ortadaki üç kısa eksen kesitinden oluşan 270 sine MR kesitini tutar. Her satır bir kesittir: "hasta" sütunu hastayı, "grup" hastanın grubunu ("NOR", "DCM" ya da "HCM"), "küme" ise hastanın kümesini verir ("eğitim", 45 hasta; "doğrulama", 15 hasta; "test", 30 hasta). "diyastol sonu" ve "sistol sonu" sütunlarının her hücresinde, değerleri 0 ile 1 arasında olan 64 × 64 piksellik bir görüntü vardır; bir piksel 2 mm'dir.  
> Görev: Bir kesitin iki karesini iki kanal olarak alan küçük bir evrişimli ağ kur. Ağ üç bloktan oluşur; her blokta görüntünün boyutunu koruyan 3 × 3 çekirdekli bir evrişim, batch normalizasyonu, ReLU ve 2 × 2 maksimum havuzlama bulunur. Bloklarda sırasıyla 8, 16 ve 32 çekirdek vardır. Blokların ardından düzleştirme, 0,3 oranında dropout ve üç grubun her birine bir skor veren tek bir doğrusal katman gelir. Ağın öğrendiği ağırlıkları, yani eğitilebilir parametrelerini say.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Eğitilebilir parametre sayısını yazdır ve "cevap" adıyla sakla.  
> Test: Ağ, ilk dört eğitim kesiti için her biri üç skordan oluşan dört satır vermeli.  

In [ ]:
#@title Adım 2.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_SETS = {"train": "eğitim", "val": "doğrulama", "test": "test"}
veri = pd.DataFrame({"hasta": P["patient"], "grup": [P["classes"][k] for k in P["y"]],
                     "küme": [_SETS[s] for s in P["set"]],
                     "diyastol sonu": list(P["X"][:, 0]), "sistol sonu": list(P["X"][:, 1])})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.2 kontrolü
teach.check_step("Adım 2.2", "cevap", globals())

In [ ]:
#@title Adım 2.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_SETS = {"train": "eğitim", "val": "doğrulama", "test": "test"}
veri = pd.DataFrame({"hasta": P["patient"], "grup": [P["classes"][k] for k in P["y"]],
                     "küme": [_SETS[s] for s in P["set"]],
                     "diyastol sonu": list(P["X"][:, 0]), "sistol sonu": list(P["X"][:, 1])})


class CmrCNN(nn.Module):
    """Üç evrişim bloğu ve bir doğrusal katman; girdi: bir kesitin iki karesi, iki kanal olarak."""

    def __init__(self, n_classes=3, dropout=0.3, batchnorm=True, in_channels=2):
        super().__init__()
        layers, c_in = [], in_channels
        for c_out in (8, 16, 32):
            layers += [nn.Conv2d(c_in, c_out, 3, padding=1)]
            layers += [nn.BatchNorm2d(c_out)] if batchnorm else []
            layers += [nn.ReLU(), nn.MaxPool2d(2)]
            c_in = c_out
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(32 * 8 * 8, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


torch.manual_seed(RANDOM_STATE)
ag = CmrCNN()
ilk4 = veri[veri["küme"] == "eğitim"].head(4)
x4 = torch.tensor(np.stack([np.stack(ilk4["diyastol sonu"]), np.stack(ilk4["sistol sonu"])], axis=1))
cevap = sum(p.numel() for p in ag.parameters() if p.requires_grad)
print(tuple(ag(x4).shape), "eğitilebilir parametre:", cevap)
teach.check_step("Adım 2.2", "cevap", globals())

**Sonuç ne gösteriyor?** Ağın 12.219 parametresi vardır; bunların yaklaşık yarısı, üçüncü bloğun 2.048 değerini üç skora çeviren son doğrusal katmandadır.

### Adım 2.3: Erken durdurmayla eğitim ve hasta başına karar

**Amaç.** Bu adımda ağ eğitilir ve bir kliniğin yapacağı gibi değerlendirilir: Her hasta için tek bir karar verilir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: ACDC'nin sine MR kesitlerinde küçük bir evrişimli ağ eğitmek ve ağı test hastalarında bir kliniğin yapacağı gibi, her hasta için tek bir kararla değerlendirmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; ACDC'deki 90 hastanın her birinin ortadaki üç kısa eksen kesitinden oluşan 270 sine MR kesitini tutar. Her satır bir kesittir: "hasta" sütunu hastayı, "grup" hastanın grubunu ("NOR", "DCM" ya da "HCM"), "küme" ise hastanın kümesini verir ("eğitim", 45 hasta; "doğrulama", 15 hasta; "test", 30 hasta). "diyastol sonu" ve "sistol sonu" sütunlarının her hücresinde, değerleri 0 ile 1 arasında olan 64 × 64 piksellik bir görüntü vardır; bir piksel 2 mm'dir.  
> Görev: Pikselleri eğitim kesitlerinin ortalaması ve standart sapmasıyla normalleştir. Bir kesitin iki karesini iki kanal olarak alan küçük bir evrişimli ağ kur. Ağ üç bloktan oluşur; her blokta görüntünün boyutunu koruyan 3 × 3 çekirdekli bir evrişim, batch normalizasyonu, ReLU ve 2 × 2 maksimum havuzlama bulunur. Bloklarda sırasıyla 8, 16 ve 32 çekirdek vardır. Blokların ardından düzleştirme, 0,3 oranında dropout ve üç grubun her birine bir skor veren tek bir doğrusal katman gelir. Ağı eğitim kesitlerinde çapraz entropi kaybıyla eğit. Her epoch'un sonunda, onlardan öğrenmeden doğrulama kesitlerindeki kaybı ölç ve bu kaybın en düşük olduğu epoch'un ağırlıklarını sakla (erken durdurma). Rastgele seçimler her çalıştırmada aynı olsun; bunun için başlangıç değeri olarak 2026 sayısını kullan. Ardından her test kesiti için üç grubun olasılıklarını hesapla, bu olasılıkların her test hastasının üç kesiti üzerinden ortalamasını al ve ortalaması en büyük grubu hastanın kararı olarak seç.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Doğrulama kaybının en düşük olduğu epoch'u ve 30 test hastasından kararı kendi grubuyla aynı olanların oranını üç ondalık basamakla yazdır; oranı "cevap" adıyla sakla.  
> Test: 30 test hastası olmalı ve her hastanın üç ortalama olasılığının toplamı 1 etmeli.  

In [ ]:
#@title Adım 2.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_SETS = {"train": "eğitim", "val": "doğrulama", "test": "test"}
veri = pd.DataFrame({"hasta": P["patient"], "grup": [P["classes"][k] for k in P["y"]],
                     "küme": [_SETS[s] for s in P["set"]],
                     "diyastol sonu": list(P["X"][:, 0]), "sistol sonu": list(P["X"][:, 1])})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.3 kontrolü
teach.check_step("Adım 2.3", "cevap", globals())

In [ ]:
#@title Adım 2.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_SETS = {"train": "eğitim", "val": "doğrulama", "test": "test"}
veri = pd.DataFrame({"hasta": P["patient"], "grup": [P["classes"][k] for k in P["y"]],
                     "küme": [_SETS[s] for s in P["set"]],
                     "diyastol sonu": list(P["X"][:, 0]), "sistol sonu": list(P["X"][:, 1])})


class CmrCNN(nn.Module):
    """Üç evrişim bloğu ve bir doğrusal katman; girdi: bir kesitin iki karesi, iki kanal olarak."""

    def __init__(self, n_classes=3, dropout=0.3, batchnorm=True, in_channels=2):
        super().__init__()
        layers, c_in = [], in_channels
        for c_out in (8, 16, 32):
            layers += [nn.Conv2d(c_in, c_out, 3, padding=1)]
            layers += [nn.BatchNorm2d(c_out)] if batchnorm else []
            layers += [nn.ReLU(), nn.MaxPool2d(2)]
            c_in = c_out
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(32 * 8 * 8, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


def train_model(model, X_tr, y_tr, X_va, y_va, epochs=40, lr=0.003, batch=16, seed=RANDOM_STATE):
    torch.manual_seed(seed)
    order_gen = torch.Generator().manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    best, best_state, history = float("inf"), None, []
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(y_tr), generator=order_gen)
        total = 0.0
        for start in range(0, len(y_tr), batch):
            j = order[start:start + batch]
            loss = F.cross_entropy(model(X_tr[j]), y_tr[j])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(j)
        model.eval()
        with torch.no_grad():
            val_loss = F.cross_entropy(model(X_va), y_va).item()
        history.append((total / len(y_tr), val_loss))
        if val_loss < best:
            best, best_state = val_loss, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return history


goruntuler = np.stack([np.stack(veri["diyastol sonu"]), np.stack(veri["sistol sonu"])], axis=1)
kodlar = veri["grup"].map({"NOR": 0, "DCM": 1, "HCM": 2}).to_numpy()
kumeler = veri["küme"].to_numpy()
tr, va, te = kumeler == "eğitim", kumeler == "doğrulama", kumeler == "test"
mean, sd = float(goruntuler[tr].mean()), float(goruntuler[tr].std())
T = lambda a: torch.tensor((a - mean) / sd, dtype=torch.float32)
torch.manual_seed(RANDOM_STATE)
ag = CmrCNN()
history = train_model(ag, T(goruntuler[tr]), torch.tensor(kodlar[tr]), T(goruntuler[va]), torch.tensor(kodlar[va]))
ag.eval()
with torch.no_grad():
    kesit_olasiligi = F.softmax(ag(T(goruntuler[te])), dim=1).numpy()
test_hastalari = veri["hasta"].to_numpy()[te]
adlar = sorted(set(test_hastalari))
ort_olasilik = np.array([kesit_olasiligi[test_hastalari == n].mean(axis=0) for n in adlar])
gercek = np.array([kodlar[te][test_hastalari == n][0] for n in adlar])
cevap = round(float((ort_olasilik.argmax(axis=1) == gercek).mean()), 3)
print("En iyi epoch:", int(np.argmin([h[1] for h in history])) + 1, "· test hastası:", len(adlar), "· doğruluk:", cevap)
teach.check_step("Adım 2.3", "cevap", globals())

**Sonuç ne gösteriyor?** Dersin çalıştırmasında doğruluk yaklaşık 0,933 olmuştur. Eğitim bilgisayardan bilgisayara biraz değiştiği için kontrol, bu değerin 0,15 yakınındaki sonuçları kabul etmektedir.

### Adım 2.4: Karışıklık matrisi ve makro F1

**Amaç.** Bu adımda sonuç sınıf sınıf okunur ve ağın kaçırdığı hastalar bulunur. Veri hücresi dersin ağını bir kez eğitir; bu yaklaşık bir dakika sürer.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: ACDC'nin 30 test hastasında bir ağın kararlarını grup grup okumak ve ağın kaçırdığı hastaları bulmak istiyorum.  
> Veri: Defterde "veri" adında bir tablo var. Her satır bir test hastasıdır: "hasta" sütunu hastanın adını, "gerçek grup" uzmanların verdiği grubu, "karar" ise ağın seçtiği grubu verir; ikisi de "NOR", "DCM" ya da "HCM" değerlerinden biridir.  
> Görev: Satırlarda gerçek gruplar, sütunlarda kararlar NOR, DCM, HCM sırasıyla yer alacak biçimde karışıklık matrisini oluştur ve yazdır. Her grubun F1 skorunu ve bunların düz ortalaması olan makro F1'i hesapla. Kararı gerçek grubundan farklı olan hastaların adlarını yazdır.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Makro F1'i üç ondalık basamakla yazdır ve "cevap" adıyla sakla.  
> Test: Karışıklık matrisinin toplamı 30, her satırının toplamı 10 olmalı.  

In [ ]:
#@title Adım 2.4 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_r = _course_decisions()
veri = pd.DataFrame({"hasta": _r["names"], "gerçek grup": [P["classes"][k] for k in _r["true"]],
                     "karar": [P["classes"][k] for k in _r["pred"]]})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 2.4 kontrolü
teach.check_step("Adım 2.4", "cevap", globals())

In [ ]:
#@title Adım 2.4 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
_r = _course_decisions()
veri = pd.DataFrame({"hasta": _r["names"], "gerçek grup": [P["classes"][k] for k in _r["true"]],
                     "karar": [P["classes"][k] for k in _r["pred"]]})
sira = ["NOR", "DCM", "HCM"]
cm = pd.crosstab(pd.Categorical(veri["gerçek grup"], sira), pd.Categorical(veri["karar"], sira),
                 rownames=["gerçek grup"], colnames=["karar"], dropna=False)
print(cm)
f1ler = []
for g in sira:
    tp = int(((veri["gerçek grup"] == g) & (veri["karar"] == g)).sum())
    fp = int(((veri["gerçek grup"] != g) & (veri["karar"] == g)).sum())
    fn = int(((veri["gerçek grup"] == g) & (veri["karar"] != g)).sum())
    f1ler.append(2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0)
print("F1:", {g: round(f, 3) for g, f in zip(sira, f1ler)})
print("Kaçırılanlar:", list(veri.loc[veri["gerçek grup"] != veri["karar"], "hasta"]))
cevap = round(float(np.mean(f1ler)), 3)
print("Makro F1:", cevap)
teach.check_step("Adım 2.4", "cevap", globals())

**Sonuç ne gösteriyor?** Makro F1 yaklaşık 0,933 olmuştur. Dersin çalıştırmasındaki iki hata, ağın normal dediği iki HCM hastasıdır; bunlardan biri 134 numaralı hastadır ve nedenini ders sayfasının son bölümü açıklamaktadır.

**2. durak tamamlandı.** [Ders sayfasına](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#stop-2) dönün ve duraktan sonrasını okumaya devam edin.

## Durak 3: Mini proje: EchoNet-Dynamic, normal mi anormal mi? (kişisel kayıtla)

*Ders: [3. uygulama durağı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#stop-3)* · *Python yolunda aynı sonuç: Kısım 2, 8-10. bölümler*

Oturumun mini projesi, her katılımcının Stanford AIMI'ye kişisel kayıtla edindiği ekokardiyografi videolarını, yani EchoNet-Dynamic'i kullanmaktadır. Kaydınız yoksa eğitmenin ortak ekrandaki çalıştırmasını izleyin. Kaydınız varsa kişisel indirme bağlantınızı defterde bu durağın ilk hücresine yapıştırın ya da zip dosyasını ders klasöründeki data/echonet klasörüne koyun. Veriler Colab oturumunuzda ya da bilgisayarınızda kalır ve hiçbir zaman bir prompta girmez: Promptlar yalnızca defterin hazırladığı tabloyu tarif eder.

**Durağın sonunda:** Kopyanızdaki video sayısını, EF'si yüzde 50'nin altında olan test videolarının oranını ve dersin ağının anormal EF için AUC değerini elde edeceksiniz. Kontroller, sonuçlarınızı aynı kopyada çalışan hazır kodla karşılaştırır.

In [ ]:
#@title 3. durağın hazırlığı: Bu hücreyi bir kez çalıştırın (durağın verilerini yükler)
# EchoNet-Dynamic için kişisel indirme bağlantınızı, yalnızca veriler henüz bu bilgisayarda değilse, tırnakların arasına yapıştırın.
ECHONET_LINK = ""

class CmrCNN(nn.Module):
    """Üç evrişim bloğu ve bir doğrusal katman; girdi: bir kesitin iki karesi, iki kanal olarak."""

    def __init__(self, n_classes=3, dropout=0.3, batchnorm=True, in_channels=2):
        super().__init__()
        layers, c_in = [], in_channels
        for c_out in (8, 16, 32):
            layers += [nn.Conv2d(c_in, c_out, 3, padding=1)]
            layers += [nn.BatchNorm2d(c_out)] if batchnorm else []
            layers += [nn.ReLU(), nn.MaxPool2d(2)]
            c_in = c_out
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(32 * 8 * 8, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))

try:
    echo = echonet.find()
except FileNotFoundError:
    echo = echonet.download(ECHONET_LINK) if ECHONET_LINK else None
if echo is None:
    print("EchoNet-Dynamic'in bir kopyası bulunamadı. Stanford AIMI'ye kaydolun, ardından kişisel bağlantınızı yukarıya yapıştırın ya da zip dosyasını ders klasöründeki data/echonet klasörüne koyun.")
else:
    print(echo)
print("3. durak hazır.")

### Adım 3.1: Videolar ve kümeleri

**Amaç.** Bu adımda her analizden önce kopyanın eksiksiz olduğu denetlenir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Her analizden önce, ekokardiyografi veri seti EchoNet-Dynamic'ten edindiğim kopyanın eksiksiz olduğunu denetlemek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; bir ekokardiyografi video veri seti olan EchoNet-Dynamic'in kendi kopyamdaki videolarını tutar. Her satır bir videodur: "video" sütunu videonun adını, "EF" ejeksiyon fraksiyonunu yüzde olarak, "küme" ise yazarların seçtiği kümeyi verir ("eğitim", "doğrulama" ya da "test").  
> Görev: Her kümedeki video sayısını ve toplam video sayısını yazdır.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Toplam video sayısını "cevap" adıyla sakla.  
> Test: "eğitim", "doğrulama" ve "test" kümelerinde sırasıyla 7.465, 1.288 ve 1.277 video bulunmalı.  

In [ ]:
#@title Adım 3.1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
if echo is None:
    print("Henüz bir EchoNet-Dynamic kopyası yok: Bu durağın ilk hücresini kişisel bağlantınızla çalıştırın ya da zip dosyasını data/echonet klasörüne koyun.")
else:
    _ft = echonet.file_list(echo)
    veri = pd.DataFrame({"video": _ft["FileName"], "EF": _ft["EF"],
                         "küme": _ft["Split"].map({"TRAIN": "eğitim", "VAL": "doğrulama", "TEST": "test"})})
    display(veri)
globals().pop("cevap", None)
pass

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.1 kontrolü
if echo is None:
    print("Henüz bir EchoNet-Dynamic kopyası yok: Bu durağın ilk hücresini kişisel bağlantınızla çalıştırın ya da zip dosyasını data/echonet klasörüne koyun.")
else:
    teach.check_value("Adım 3.1", "cevap", 10030, 0, globals())

In [ ]:
#@title Adım 3.1 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
if echo is None:
    print("Henüz bir EchoNet-Dynamic kopyası yok: Bu durağın ilk hücresini kişisel bağlantınızla çalıştırın ya da zip dosyasını data/echonet klasörüne koyun.")
else:
    _ft = echonet.file_list(echo)
    veri = pd.DataFrame({"video": _ft["FileName"], "EF": _ft["EF"],
                         "küme": _ft["Split"].map({"TRAIN": "eğitim", "VAL": "doğrulama", "TEST": "test"})})
    display(veri)
    print(veri["küme"].value_counts())
    cevap = int(len(veri))
    print("Video:", cevap)
    teach.check_value("Adım 3.1", "cevap", 10030, 0, globals())

**Sonuç ne gösteriyor?** Eksiksiz bir kopya 10.030 video içerir; yazarların ayırmasında bunların 7.465'i eğitim, 1.288'i doğrulama, 1.277'si de test kümesindedir.

### Adım 3.2: Anormal EF oranı

**Amaç.** Bu adımda bir model değerlendirilmeden önce anormal sınıfın ne kadar sık olduğu öğrenilir.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: EchoNet-Dynamic için bir model değerlendirilmeden önce, test videoları arasında yüzde 50'nin altındaki anormal bir ejeksiyon fraksiyonunun ne kadar sık olduğunu öğrenmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; bir ekokardiyografi video veri seti olan EchoNet-Dynamic'in kendi kopyamdaki videolarını tutar. Her satır bir videodur: "video" sütunu videonun adını, "EF" ejeksiyon fraksiyonunu yüzde olarak, "küme" ise yazarların seçtiği kümeyi verir ("eğitim", "doğrulama" ya da "test").  
> Görev: Kümesi "test" olan videoları seç ve bunlar arasında EF'si yüzde 50'nin altında olanların oranını hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Oranı üç ondalık basamakla yazdır ve "cevap" adıyla sakla.  
> Test: Oran 0 ile 1 arasında olmalı ve test videosu sayısı 1.277 olmalı.  

In [ ]:
#@title Adım 3.2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
if echo is None:
    print("Henüz bir EchoNet-Dynamic kopyası yok: Bu durağın ilk hücresini kişisel bağlantınızla çalıştırın ya da zip dosyasını data/echonet klasörüne koyun.")
else:
    _ft = echonet.file_list(echo)
    veri = pd.DataFrame({"video": _ft["FileName"], "EF": _ft["EF"],
                         "küme": _ft["Split"].map({"TRAIN": "eğitim", "VAL": "doğrulama", "TEST": "test"})})
    display(veri)
globals().pop("cevap", None)
pass

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.2 kontrolü
if echo is None:
    print("Henüz bir EchoNet-Dynamic kopyası yok: Bu durağın ilk hücresini kişisel bağlantınızla çalıştırın ya da zip dosyasını data/echonet klasörüne koyun.")
else:
    teach.check_value("Adım 3.2", "cevap", round(float((echonet.file_list(echo).query('Split == "TEST"')["EF"] < 50).mean()), 3), 0.001, globals())

In [ ]:
#@title Adım 3.2 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
if echo is None:
    print("Henüz bir EchoNet-Dynamic kopyası yok: Bu durağın ilk hücresini kişisel bağlantınızla çalıştırın ya da zip dosyasını data/echonet klasörüne koyun.")
else:
    _ft = echonet.file_list(echo)
    veri = pd.DataFrame({"video": _ft["FileName"], "EF": _ft["EF"],
                         "küme": _ft["Split"].map({"TRAIN": "eğitim", "VAL": "doğrulama", "TEST": "test"})})
    display(veri)
    test_tablosu = veri[veri["küme"] == "test"]
    cevap = round(float((test_tablosu["EF"] < 50).mean()), 3)
    print(len(test_tablosu), "test videosu; EF'si yüzde 50'nin altında olanların oranı:", cevap)
    teach.check_value("Adım 3.2", "cevap", round(float((echonet.file_list(echo).query('Split == "TEST"')["EF"] < 50).mean()), 3), 0.001, globals())

**Sonuç ne gösteriyor?** Anormal sınıf küçük olandır. Her videoya normal diyen bir model, bir eksi bu oran kadar doğruluğa ulaşırdı; durağın AUC değerini ölçmesinin nedeni budur.

### Adım 3.3: Dersin ağı ekokardiyografide

**Amaç.** Bu adımda aynı ağ videoların diyastol sonu ve sistol sonu karelerinde eğitilir ve anormal EF'yi ne kadar iyi tanıdığı ölçülür. Veri hücresi 1.400 videoyu okur; bu birkaç dakika sürer.

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Küçük bir evrişimli ağın, bir ekokardiyografi videosunun iki karesinden, yani diyastol sonu ve sistol sonu karelerinden yüzde 50'nin altındaki bir ejeksiyon fraksiyonunu ne kadar iyi tanıdığını öğrenmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; EchoNet-Dynamic'ten 1.000 eğitim ve 400 test videosunu tutar. Her satır bir videodur: "küme" sütununda "eğitim" ya da "test" yazar; "anormal" sütunu EF yüzde 50'nin altındaysa 1, değilse 0'dır. "diyastol sonu" ve "sistol sonu" sütunlarının her hücresi, o fazın çizilmiş karesini değerleri 0 ile 1 arasında olan 64 × 64 piksellik bir görüntü olarak tutar.  
> Görev: Pikselleri eğitim videolarının ortalaması ve standart sapmasıyla normalleştir. Bir videonun iki karesini iki kanal olarak alan küçük bir evrişimli ağ kur. Ağ üç bloktan oluşur; her blokta görüntünün boyutunu koruyan 3 × 3 çekirdekli bir evrişim, batch normalizasyonu, ReLU ve 2 × 2 maksimum havuzlama bulunur. Bloklarda sırasıyla 8, 16 ve 32 çekirdek vardır. Blokların ardından düzleştirme, 0,3 oranında dropout ve normal EF ile anormal EF için birer skor veren tek bir doğrusal katman gelir. Ağı eğitim videolarında çapraz entropi kaybıyla eğit. Rastgele seçimler her çalıştırmada aynı olsun; bunun için başlangıç değeri olarak 2026 sayısını kullan. Ardından her test videosu için anormal EF olasılığını ve ROC eğrisinin altındaki alanı (AUC) hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: AUC değerini üç ondalık basamakla yazdır ve "cevap" adıyla sakla.  
> Test: İki karesi 64 × 64 piksel olan 1.000 eğitim videosu bulunmalı ve AUC 0,5 ile 1 arasında olmalı.  

In [ ]:
#@title Adım 3.3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
if echo is None:
    print("Henüz bir EchoNet-Dynamic kopyası yok: Bu durağın ilk hücresini kişisel bağlantınızla çalıştırın ya da zip dosyasını data/echonet klasörüne koyun.")
else:
    _tr = echonet.frame_pairs(echo, split="TRAIN", n=1000, size=64, seed=RANDOM_STATE)
    _te = echonet.frame_pairs(echo, split="TEST", n=400, size=64, seed=RANDOM_STATE)
    veri = pd.DataFrame({"küme": ["eğitim"] * len(_tr["abnormal"]) + ["test"] * len(_te["abnormal"]),
                         "anormal": np.concatenate([_tr["abnormal"], _te["abnormal"]]).astype(int),
                         "diyastol sonu": list(np.concatenate([_tr["images"][:, 0], _te["images"][:, 0]])),
                         "sistol sonu": list(np.concatenate([_tr["images"][:, 1], _te["images"][:, 1]]))})
    display(veri)
globals().pop("cevap", None)
pass

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

In [ ]:
# Adım 3.3 kontrolü
if echo is None:
    print("Henüz bir EchoNet-Dynamic kopyası yok: Bu durağın ilk hücresini kişisel bağlantınızla çalıştırın ya da zip dosyasını data/echonet klasörüne koyun.")
else:
    teach.check_range("Adım 3.3", "cevap", 0.6, 1.0, globals())

In [ ]:
#@title Adım 3.3 için hazır kod (yalnızca asistanın kodu kontrolden geçmezse çalıştırın)
if echo is None:
    print("Henüz bir EchoNet-Dynamic kopyası yok: Bu durağın ilk hücresini kişisel bağlantınızla çalıştırın ya da zip dosyasını data/echonet klasörüne koyun.")
else:
    _tr = echonet.frame_pairs(echo, split="TRAIN", n=1000, size=64, seed=RANDOM_STATE)
    _te = echonet.frame_pairs(echo, split="TEST", n=400, size=64, seed=RANDOM_STATE)
    veri = pd.DataFrame({"küme": ["eğitim"] * len(_tr["abnormal"]) + ["test"] * len(_te["abnormal"]),
                         "anormal": np.concatenate([_tr["abnormal"], _te["abnormal"]]).astype(int),
                         "diyastol sonu": list(np.concatenate([_tr["images"][:, 0], _te["images"][:, 0]])),
                         "sistol sonu": list(np.concatenate([_tr["images"][:, 1], _te["images"][:, 1]]))})
    display(veri)
    class CmrCNN(nn.Module):
        """Üç evrişim bloğu ve bir doğrusal katman; girdi: bir kesitin iki karesi, iki kanal olarak."""

        def __init__(self, n_classes=3, dropout=0.3, batchnorm=True, in_channels=2):
            super().__init__()
            layers, c_in = [], in_channels
            for c_out in (8, 16, 32):
                layers += [nn.Conv2d(c_in, c_out, 3, padding=1)]
                layers += [nn.BatchNorm2d(c_out)] if batchnorm else []
                layers += [nn.ReLU(), nn.MaxPool2d(2)]
                c_in = c_out
            self.features = nn.Sequential(*layers)
            self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(32 * 8 * 8, n_classes))

        def forward(self, x):
            return self.classifier(self.features(x))


    goruntuler = np.stack([np.stack(veri["diyastol sonu"]), np.stack(veri["sistol sonu"])], axis=1)
    egitim = (veri["küme"] == "eğitim").to_numpy()
    etiketler = veri["anormal"].to_numpy()
    mean, sd = float(goruntuler[egitim].mean()), float(goruntuler[egitim].std())
    X_tr = torch.tensor((goruntuler[egitim] - mean) / sd, dtype=torch.float32)
    X_te = torch.tensor((goruntuler[~egitim] - mean) / sd, dtype=torch.float32)
    y_tr = torch.tensor(etiketler[egitim], dtype=torch.long)
    torch.manual_seed(2026)
    eko_agi = CmrCNN(n_classes=2)
    optimizer = torch.optim.Adam(eko_agi.parameters(), lr=0.003)
    for epoch in range(15):
        eko_agi.train()
        order = torch.randperm(len(y_tr))
        for start in range(0, len(y_tr), 32):
            j = order[start:start + 32]
            loss = F.cross_entropy(eko_agi(X_tr[j]), y_tr[j])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    eko_agi.eval()
    with torch.no_grad():
        p_anormal = F.softmax(eko_agi(X_te), dim=1)[:, 1].numpy()
    cevap = round(float(roc_auc_score(etiketler[~egitim], p_anormal)), 3)
    print(tuple(X_tr.shape), "AUC:", cevap)
    teach.check_range("Adım 3.3", "cevap", 0.6, 1.0, globals())

**Sonuç ne gösteriyor?** 64 × 64 piksellik iki kare bile EF hakkında önemli bir bilgi taşır; çünkü kavitenin diyastol sonu ile sistol sonu arasındaki değişimi görünmektedir. Sonuç kopyadan kopyaya ve bilgisayardan bilgisayara değiştiği için kontrol, 0,6 ile 1,0 arasındaki bir AUC değerini kabul eder. EchoNet-Dynamic ise videonun tamamını daha yüksek çözünürlükte okur ve azalmış EF için 0,97'lik bir AUC değerine ulaşmıştır.

**3. durak tamamlandı.** [Ders sayfasına](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#stop-3) dönün ve duraktan sonrasını okumaya devam edin.

## Prompt yolunda haftalık proje

*Haftanın projesi: [3. haftanın README sayfası, Haftalık proje](https://github.com/utkukose/ai-cardiology-tkd-NB-lecture/blob/main/tr/hafta-03/README.md#haftal%C4%B1k-proje)*

Haftalık proje, her adım için bir prompt olmak üzere aşağıdaki promptlarla da yapılabilir. Önce hazırlık hücresini bir kez çalıştırın; hücre projenin verilerini yükler. Ardından her adım, duraklardaki adımlar gibi ilerler. Tek fark şudur: Projede kontrol hücresi ve hazır kod yoktur, çünkü sonuçlar rapora girer. Bir promptun testi sağlanmadan sonraki prompta geçilmez ve her prompt YZ kullanım günlüğüne yazılır.

In [ ]:
#@title Projenin hazırlığı: Bu hücreyi bir kez çalıştırın (projenin verilerini yükler)
GROUPS = ("NOR", "DCM", "HCM")
P = acdc.pairs(GROUPS)
acdc_table = acdc.table(GROUPS)
_SETS = {"train": "eğitim", "val": "doğrulama", "test": "test"}

class CmrCNN(nn.Module):
    """Üç evrişim bloğu ve bir doğrusal katman; girdi: bir kesitin iki karesi, iki kanal olarak."""

    def __init__(self, n_classes=3, dropout=0.3, batchnorm=True, in_channels=2):
        super().__init__()
        layers, c_in = [], in_channels
        for c_out in (8, 16, 32):
            layers += [nn.Conv2d(c_in, c_out, 3, padding=1)]
            layers += [nn.BatchNorm2d(c_out)] if batchnorm else []
            layers += [nn.ReLU(), nn.MaxPool2d(2)]
            c_in = c_out
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(32 * 8 * 8, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


def train_model(model, X_tr, y_tr, X_va, y_va, epochs=40, lr=0.003, batch=16, seed=RANDOM_STATE):
    torch.manual_seed(seed)
    order_gen = torch.Generator().manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    best, best_state, history = float("inf"), None, []
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(y_tr), generator=order_gen)
        total = 0.0
        for start in range(0, len(y_tr), batch):
            j = order[start:start + batch]
            loss = F.cross_entropy(model(X_tr[j]), y_tr[j])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(j)
        model.eval()
        with torch.no_grad():
            val_loss = F.cross_entropy(model(X_va), y_va).item()
        history.append((total / len(y_tr), val_loss))
        if val_loss < best:
            best, best_state = val_loss, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return history



def _project_course():
    """Dersin ağı; bir kez eğitilir (yaklaşık bir dakika) ve test kesitlerinin olasılıklarını verir."""
    if "_project_result" in globals():
        return _project_result
    X, y, sets, patients = P["X"], P["y"], P["set"], P["patient"]
    tr, va, te = sets == "train", sets == "val", sets == "test"
    m, sd = float(X[tr].mean()), float(X[tr].std())
    T = lambda a: torch.tensor((a - m) / sd, dtype=torch.float32)
    torch.manual_seed(RANDOM_STATE)
    net = CmrCNN()
    train_model(net, T(X[tr]), torch.tensor(y[tr]), T(X[va]), torch.tensor(y[va]))
    net.eval()
    with torch.no_grad():
        prob = F.softmax(net(T(X[te])), dim=1).numpy()
    names = sorted(set(patients[te]))
    mean_prob = np.array([prob[patients[te] == n].mean(axis=0) for n in names])
    globals()["_project_result"] = {"prob": prob, "names": names, "pred": mean_prob.argmax(axis=1),
                                    "true": np.array([y[te][patients[te] == n][0] for n in names])}
    return _project_result


def _project_middle_slices(name):
    """Bir hastanın ortadaki üç kesiti, yani ağın gördüğü kesitler."""
    d = acdc.default_folder() / name
    gt, _ = read_nifti(d / f"{name}_frame{int(acdc.read_info(d / 'Info.cfg')['ED']):02d}_gt.nii.gz")
    lv = [k for k in range(gt.shape[2]) if (gt[:, :, k] == 3).sum() > 0]
    mid = lv[len(lv) // 2]
    return sorted(sorted(lv, key=lambda k: (abs(k - mid), k))[:3])
print("Proje hazır.")

### Proje adımı 1: Ağı eğitmek

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: ACDC'nin sine MR kesitlerinde küçük bir evrişimli ağı erken durdurmayla eğitmek ve eğitim ile doğrulama kesitlerinin kayıplarının nasıl değiştiğini görmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; ACDC'deki 90 hastanın her birinin ortadaki üç kısa eksen kesitinden oluşan 270 sine MR kesitini tutar. Her satır bir kesittir: "hasta" sütunu hastayı, "grup" hastanın grubunu ("NOR", "DCM" ya da "HCM"), "küme" ise hastanın kümesini verir ("eğitim", 45 hasta; "doğrulama", 15 hasta; "test", 30 hasta). "diyastol sonu" ve "sistol sonu" sütunlarının her hücresinde, değerleri 0 ile 1 arasında olan 64 × 64 piksellik bir görüntü vardır; bir piksel 2 mm'dir.  
> Görev: Pikselleri eğitim kesitlerinin ortalaması ve standart sapmasıyla normalleştir. Bir kesitin iki karesini iki kanal olarak alan küçük bir evrişimli ağ kur. Ağ üç bloktan oluşur; her blokta görüntünün boyutunu koruyan 3 × 3 çekirdekli bir evrişim, batch normalizasyonu, ReLU ve 2 × 2 maksimum havuzlama bulunur. Bloklarda sırasıyla 8, 16 ve 32 çekirdek vardır. Blokların ardından düzleştirme, 0,3 oranında dropout ve üç grubun her birine bir skor veren tek bir doğrusal katman gelir. Ağı eğitim kesitlerinde çapraz entropi kaybıyla eğit. Rastgele seçimler her çalıştırmada aynı olsun; bunun için başlangıç değeri olarak 2026 sayısını kullan. Her epoch'un sonunda, onlardan öğrenmeden doğrulama kesitlerindeki kaybı ölç ve bu kaybın en düşük olduğu epoch'un ağırlıklarını sakla. Eğitim ve doğrulama kaybını epoch'lara karşı çiz.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: İki kaybın şeklini göster; doğrulama kaybının en düşük olduğu epoch'u "cevap" adıyla sakla.  
> Test: Saklanan ağırlıklar yüklendikten sonra doğrulama kaybı yine en düşük değerini vermeli.  

In [ ]:
#@title Proje adımı 1 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
veri = pd.DataFrame({"hasta": P["patient"], "grup": [P["classes"][k] for k in P["y"]],
                     "küme": [_SETS[s] for s in P["set"]],
                     "diyastol sonu": list(P["X"][:, 0]), "sistol sonu": list(P["X"][:, 1])})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 2: Ağın karışıklık matrisi

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: ACDC'nin 30 test hastasında, her hasta için tek bir kararla bir ağın karışıklık matrisini ve her grubun kesinlik, duyarlılık ve F1 değerlerini görmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; 30 test hastasının her birinden üç kesit olmak üzere 90 test kesitini tutar. Her satır bir kesittir: "hasta" sütunu hastayı, "gerçek grup" uzmanların verdiği grubu, "NOR", "DCM" ve "HCM" sütunları da dersin ağının bu kesite verdiği üç grubun olasılıklarını 0 ile 1 arasında bir sayı olarak verir. Veri hücresi bu ağı bir kez eğitir; bu yaklaşık bir dakika sürer.  
> Görev: Olasılıkların her hastanın kesitleri üzerinden ortalamasını al ve ortalaması en büyük grubu karar olarak seç. 30 hastanın karışıklık matrisini, satırlarda gerçek gruplar ve sütunlarda kararlar NOR, DCM, HCM sırasıyla yer alacak biçimde oluştur. Her grubun kesinlik, duyarlılık ve F1 değerini ve makro F1'i hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Matrisi ve gruplara göre üç ölçütün tablosunu göster; makro F1'i "cevap" adıyla sakla.  
> Test: Matrisin toplamı 30, her satırının toplamı 10 olmalı.  

In [ ]:
#@title Proje adımı 2 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_r = _project_course()
_te = P["set"] == "test"
veri = pd.DataFrame({"hasta": P["patient"][_te], "gerçek grup": [P["classes"][k] for k in P["y"][_te]],
                     "NOR": _r["prob"][:, 0], "DCM": _r["prob"][:, 1], "HCM": _r["prob"][:, 2]})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 3: Aynı hastalarda ACDC kuralları

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Ağı, aynı 30 test hastasında ACDC'nin ölçüm kurallarıyla karşılaştırmak istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; ACDC'nin NOR (normal), DCM (dilate kardiyomiyopati) ve HCM (hipertrofik kardiyomiyopati) gruplarındaki 30 test hastasını tutar. Her satır bir hastadır: "hasta" sütunu hastanın adını, "grup" grubunu, "LVEF" sol ventrikül ejeksiyon fraksiyonunu yüzde olarak, "LVEDVI" vücut yüzey alanına göre indekslenmiş sol ventrikül diyastol sonu hacmini mL/m² olarak, "en kalın duvar" da diyastol sonunda duvarın en kalın segmentini mm olarak verir. Bütün değerler uzmanların çizdiği konturlarda ölçülmüştür.  
> Görev: ACDC kurallarını uygula: LVEF 40'ın altında ve LVEDVI 100'ün üzerindeyse hasta DCM'dir; değilse en kalın duvar 15'in üzerinde ve LVEF 50 ya da daha yüksekse HCM'dir; ikisi de değilse NOR'dur. Grup ile kuralın karışıklık matrisini NOR, DCM, HCM sırasıyla oluştur ve doğruluğu hesapla.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Matrisi göster ve doğruluğu üç ondalık basamakla yazdır; doğruluğu "cevap" adıyla sakla.  
> Test: Matrisin toplamı 30 olmalı.  

In [ ]:
#@title Proje adımı 3 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_t = acdc_table[acdc_table["split"] == "testing"]
veri = _t[["patient", "group", "lv_ef", "lv_edvi", "max_wall_mm"]].rename(columns={
    "patient": "hasta", "group": "grup", "lv_ef": "LVEF", "lv_edvi": "LVEDVI", "max_wall_mm": "en kalın duvar"})
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

### Proje adımı 4: Hatalar

**Prompt.** Aşağıdaki metni kopyalayın ve YZ asistanınıza gönderin. Promptun sözünü ettiği "veri" tablosunu hemen altındaki veri hücresi hazırlar.

> Rol: Kod yazmayan bir kardiyoloğa yardım ediyorsun. Yazacağın kodu Jupyter defterime yapıştırıp çalıştıracağım.  
> Bağlam: Bir ACDC ağının yanlış sınıfladığı test hastalarında, sol ventrikülün en kalın duvarının tabandan apekse doğru nerede bulunduğunu görmek istiyorum.  
> Veri: Defterde "veri" adında bir tablo var; dersin ağının yanlış sınıfladığı test hastalarını tutar. Her satır bu hastalardan birinin diyastol sonundaki bir kısa eksen kesitidir: "hasta" sütunu hastayı, "gerçek grup" ve "karar" uzmanların ve ağın verdiği grubu, "kesit" kesitin tabandan (0) apekse doğru numarasını, "en kalın duvar" o kesitteki duvarın en kalın segmentini mm olarak verir. "ağın gördüğü" sütununda, ağın gördüğü tek kesitler olan ortadaki üç kesit için "evet" yazar.  
> Görev: Her hasta için en kalın duvarı kesit numarasına karşı çiz ve ağın gördüğü kesitleri işaretle. İncelemenin en kalın duvarının bulunduğu kesiti ve ağın bu kesiti görüp görmediğini yazdır.  
> Kısıtlar: Yalnızca bu tabloyu kullan. Kod dosya okumamalı, internetten bir şey indirmemeli ve program kurmamalıdır.  
> Çıktı: Her hasta için bir şekil göster; yanlış sınıflanan hasta sayısını "cevap" adıyla sakla.  
> Test: Her hastada "ağın gördüğü" sütununda tam üç "evet" bulunmalı; hasta sayısı da proje adımı 2'deki karışıklık matrisinin hata sayısına eşit olmalı.  

In [ ]:
#@title Proje adımı 4 için veri: Bu hücreyi çalıştırın ("veri" tablosunu hazırlar ve gösterir)
_r = _project_course()
_rows = []
for _n, _t, _p in zip(_r["names"], _r["true"], _r["pred"]):
    if _t == _p:
        continue
    _seen = _project_middle_slices(_n)
    for _k, _w in acdc.walls(_n).items():
        _rows.append({"hasta": _n, "gerçek grup": P["classes"][_t], "karar": P["classes"][_p],
                      "kesit": _k, "en kalın duvar": _w,
                      "ağın gördüğü": "evet" if _k in _seen else "hayır"})
veri = pd.DataFrame(_rows)
globals().pop("cevap", None)
veri

In [ ]:
# YZ asistanının kodunu bu satırın yerine yapıştırın ve hücreyi çalıştırın.

<a id="part-2"></a>

# Kısım 2: Python yolu

**Kimler için?** Bu kısım Python yazmayı öğrenmek isteyen katılımcılar içindir. Kısım 1'i çalışmış olmanız gerekmez: 0. bölümden sonra doğrudan buradan başlayın.

**Nasıl ilerler?** Hücreleri yukarıdan aşağıya Shift+Enter ile çalıştırın. Baştaki *Python adımı*, haftanın gerektirdiği dil bilgisini öğretir. Numaralı bölümlerdeki alıştırmalar hemen *doğru* ya da *henüz değil* yazar; örnek çözümler katlanmış hücrelerde durur ve ancak denedikten sonra açılmalıdır. *Gözlemle ve düşün* hücreleri açık uçlu sorular sorar; bu soruların çözüm hücresi yoktur. *Python notu* hücreleri, yeni bir yapının ilk kullanıldığı yerde dili açıklar. *İleri düzey (isteğe bağlı)* olarak işaretlenen hücreler, programlama bilenler için ek çalışmalardır. Bu kısma ortadan başlayan katılımcı, o bölümün başındaki *Telafi* hücresini çalıştırır. Bazı alıştırma hücreleri `veri` adlı bir tabloyu hazırlayan bir satırla başlar. Bu tablo, etkileşimli labın prompt atölyesindeki alıştırma promptları için hazırlanmıştır; Python yolunda alıştırma, metninde adı geçen tablolarla çözülebilir.

## Python adımı 3: Görüntüleri dizi olarak işlemek

Bilgisayar için gri tonlu bir görüntü, her piksel için bir sayı içeren bir tablodur. Üst üste konmuş görüntüler üç boyutlu bir dizi oluşturur; bir sine inceleme buna zamanı da ekler. Bu adımda elle bir görüntü oluşturulur, ACDC'nin bir sine kardiyak manyetik rezonans hacmi dosyasından okunur, bir görüntü dizisi dilimlenir ve normalleştirilir; sonunda da tek bir nöronun eğitim döngüsü yazılır. Defterin sonraki bütün bölümleri bu adımları kullanmaktadır.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
img7 = np.zeros((7, 7))
img7[1:6, 1:6] = 0.2      # miyokard gibi koyu bir halka
img7[2:5, 2:5] = 1.0      # kavitedeki kan gibi parlak bir merkez
print(img7.shape, img7.dtype)
print(img7)
plt.figure(figsize=(2.2, 2.2))
plt.imshow(img7, cmap="gray", vmin=0, vmax=1)
plt.axis("off")
plt.show()

**Python notu.** `np.zeros((7, 7))` 0 ile dolu, 7 satır ve 7 sütunluk bir dizi oluşturur. `img7[1:6, 1:6]` 1-5. satırları ve 1-5. sütunları seçer: Bir `a:b` dilimine ilk sayı dâhildir, ikincisi dâhil değildir. İlk indis her zaman yukarıdan sayılan satır, ikincisi sütundur. `plt.imshow` diziyi çizer; `vmin` ve `vmax` hangi değerlerin siyah ve beyaz olacağını sabitler. Sine kardiyak manyetik rezonansta kan parlak, miyokard koyu görünür.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
case_dir = acdc.default_folder() / "patient134"
info = acdc.read_info(case_dir / "Info.cfg")
print(info)
vol, spacing = read_nifti(case_dir / f"patient134_frame{int(info['ED']):02d}.nii.gz")
print(vol.shape, vol.dtype, spacing)
mid = vol.shape[2] // 2
img = vol[:, :, mid].T.astype(float)
plt.figure(figsize=(3, 3.4))
plt.imshow(img, cmap="gray")
plt.title(f"patient134, ED, kesit {mid}")
plt.axis("off")
plt.show()

**Python notu.** `acdc.read_info`, küçük Info.cfg metin dosyasını bir sözlüğe okur; sözlükte grup, boy, ağırlık, kare sayısı ve diyastol sonu (ED) ile sistol sonu (ES) kareleri bulunur. `read_nifti`, hacmin voksellerini ve vokselin mm cinsinden boyutlarını döndürür. Hacmin üç boyutu vardır (x, y, kesit); `vol[:, :, mid]` tek bir kesiti alır. `.T` satırlarla sütunların yerini değiştirir; böylece görüntü, cihazın satırları yukarıdan aşağıya gelecek biçimde görünür. `//` tam sayı bölmesi yapar ve kalanı atar.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
S = acdc.slices(GROUPS)
print(S["images"].shape, S["classes"], S["pixel_mm"], "piksel başına mm")
X12 = S["images"][:12]
print(X12.shape, S["patient"][:6], S["phase"][:6], S["group"][:6])
print(X12[0].shape, X12[:, 32, :].shape, round(float(X12[0, 32, 32]), 3))

**Python notu.** `acdc.slices`, her hastanın ortadaki üç kesitinde, diyastol sonu ve sistol sonu karelerinden sol ventrikülü ortalayan 128 mm'lik bir kare keser ve bu kareyi 64 × 64 piksele yeniden örnekler. Sonuç bir sözlüktür: `S["images"]` 540 görüntü tutar; `S["patient"]`, `S["phase"]` ve `S["group"]` ise her görüntünün hangi hastadan, hangi fazdan ve hangi gruptan geldiğini gösterir. `X12[0]` ilk görüntüdür; `X12[:, 32, :]` ise her görüntünün 32. satırını alır. Tek başına iki nokta *hepsi* anlamına gelir.

In [ ]:
# Python alıştırması: dili öğrenmek için küçük bir örnek
x = X12[0]
z = (x - x.mean()) / x.std()                      # z skorları: ortalama 0, SS 1
m = (x - x.min()) / (x.max() - x.min())           # en küçük-en büyük: 0'dan 1'e
print(round(float(z.mean()), 3), round(float(z.std()), 3), round(float(m.min()), 3), round(float(m.max()), 3))
crop = x[16:48, 16:48]
print("kırpma", crop.shape)

**Python notu.** Bir dizi üzerindeki aritmetik işlemler her elemana aynı anda uygulanır; böylece `x - x.mean()` ortalamayı 4.096 pikselin hepsinden çıkarır. Bir ağ, 0 çevresinde toplanan ve yayılımı 1'e yakın olan girdilerden daha kolay öğrenir. Bir ağ eğitilirken ortalama ve standart sapma (SS), 3. bölümde görüleceği gibi yalnızca eğitim görüntülerinden hesaplanmalıdır. `x[16:48, 16:48]` ortadaki 32 × 32 pikseli kırpar.

In [ ]:
# Python alıştırması: tek bir nöronun eğitim döngüsü
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


acdc_table = acdc.table(GROUPS)
feats = acdc_table[["lv_edvi", "max_wall_mm"]].to_numpy(float)
target = (acdc_table["group"] != "NOR").astype(int).to_numpy()     # 1: DCM ya da HCM
train_rows = (acdc_table["split"] == "training").to_numpy()
mu, sd = feats[train_rows].mean(axis=0), feats[train_rows].std(axis=0)
Xs = (feats - mu) / sd
Xn, tn = Xs[train_rows], target[train_rows]

w, b, lr = np.zeros(2), 0.0, 0.5
for step in range(201):
    p = sigmoid(Xn @ w + b)                                     # 1. ileri geçiş
    loss = -np.mean(tn * np.log(p) + (1 - tn) * np.log(1 - p))  # 2. kayıp
    grad_w = Xn.T @ (p - tn) / len(tn)                          # 3. gradyanlar
    grad_b = np.mean(p - tn)
    if step % 50 == 0:
        print(step, round(loss, 4), np.round(w, 3), round(b, 3))
    w = w - lr * grad_w                                         # 4. güncelleme
    b = b - lr * grad_b

**Python notu.** Her eğitim döngüsünün aynı dört adımı vardır: *İleri geçiş* tahminleri hesaplar, *kayıp* bunların ne kadar yanlış olduğunu ölçer, *gradyanlar* kaybı azaltmak için her ağırlığın hangi yöne gitmesi gerektiğini gösterir, *güncelleme* ise ağırlıkları bu yönde, öğrenme hızıyla ölçeklenmiş küçük bir adım ilerletir. Nöron, 60 eğitim hastasının iki standartlaştırılmış ölçümünden bir kalbin normal olup olmadığını, yani kardiyomiyopati bulunup bulunmadığını öğrenir. `Xn @ w`, 60 × 2'lik tabloyu iki ağırlıkla çarpar. `for step in range(201)` girintili satırları 201 kez tekrarlar; `step % 50 == 0` her 50 adımda bir doğru olur.

**Hızlı kontrol.** `X12` dizisinin boyutu (12, 64, 64)'tür. `X12[:, 10:20, 10:20]` dizisinin boyutu nedir?

<details><summary>Cevap</summary>

(12, 10, 10): 12 görüntünün hepsi, 10-19. satırlar ve 10-19. sütunlar.

</details>

**Alıştırma 3.1.** 134 numaralı hastanın diyastol sonu hacmi olan `vol` toplam kaç voksel tutmaktadır?

In [ ]:
veri = alistirma_tablosu("3.1")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma 3.1", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = int(vol.size)
teach.check("Alıştırma 3.1", cevap)

**Alıştırma 3.2.** `vol` hacminin ortadaki kesiti olan `img` görüntüsündeki piksellerin ortalama değeri nedir? Sonucu bir ondalık basamağa yuvarlayın.

In [ ]:
veri = alistirma_tablosu("3.2")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma 3.2", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = round(float(img.mean()), 1)
teach.check("Alıştırma 3.2", cevap)

**Alıştırma 3.3.** Nöronu sıfır ağırlıklardan başlayarak, bu kez 0,5 yerine 0,05 öğrenme hızıyla yeniden eğitin. 200 güncellemeden sonra kayıp nedir? Sonucu üç ondalık basamağa yuvarlayın.

In [ ]:
veri = alistirma_tablosu("3.3")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma 3.3", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
w3, b3 = np.zeros(2), 0.0
for step in range(200):
    p3 = sigmoid(Xn @ w3 + b3)
    w3 = w3 - 0.05 * Xn.T @ (p3 - tn) / len(tn)
    b3 = b3 - 0.05 * np.mean(p3 - tn)
p3 = sigmoid(Xn @ w3 + b3)
cevap = round(float(-np.mean(tn * np.log(p3) + (1 - tn) * np.log(1 - p3))), 3)
teach.check("Alıştırma 3.3", cevap)

## 1. NumPy ile bir nöron ve bir ağ

*Ders: [Nöron, katman, aktivasyon fonksiyonu (tıbbi analoji)](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#neural-networks-in-one-page)*

Bir nöron, girdilerinin her birini kendi ağırlığıyla çarpıp toplar, bir sabit terim ekler ve toplamı bir *aktivasyon fonksiyonundan* geçirir. Bir katman, aynı girdileri alan nöronlardan oluşur; bir ağ ise katmanları üst üste dizer ve her katman bir öncekinin çıktıları üzerinde çalışır. Doğrusal olmayan bir aktivasyon olmasaydı, üst üste dizilen katmanlar tek bir düz sınıra indirgenirdi.

**Formüllerden koda.** Ağırlıklı toplam $z = \mathbf{w} \cdot \mathbf{x} + b$, `w @ x + b` ile hesaplanır. Sigmoid $\sigma(z) = 1/(1 + e^{-z})$ `sigmoid(z)` ile, düzeltilmiş doğrusal birim (ReLU) $\max(0, z)$ ise `np.maximum(0, z)` ile yazılır; softmax $e^{z_k}/\sum_j e^{z_j}$ ise son katmanın çıktılarını toplamı 1 olan olasılıklara çevirir. Ağırlık matrisi `W1` (girdi sayısı × nöron sayısı) boyutunda olan bir katman, bütün nöronları için `x @ W1 + b1` değerini tek seferde hesaplar.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 1. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB03_derin_ogrenme_ve_evrisimli_aglar.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(1)

In [ ]:
# Üç aktivasyon fonksiyonu
zz = np.linspace(-5, 5, 201)
fig, ax = plt.subplots(figsize=(6.2, 3.2))
ax.plot(zz, sigmoid(zz), color="#9B1B1B", lw=2, label="sigmoid")
ax.plot(zz, np.tanh(zz), color="#1F5F8B", lw=2, label="tanh")
ax.plot(zz, np.maximum(0, zz), color="#B97813", lw=2, label="ReLU")
ax.set_ylim(-1.2, 3)
ax.axhline(0, color="#888888", lw=0.8)
ax.set_xlabel("z")
ax.legend()
plt.show()

In [ ]:
# Küçük bir ağ: 2 girdi, ReLU kullanan 3 gizli nöron, softmax kullanan 2 çıktı
def softmax(v):
    e = np.exp(v - v.max())
    return e / e.sum()


W1 = np.array([[1.5, -1.0, 0.5],
               [0.2, 1.2, -0.8]])
b1 = np.array([0.0, -0.5, 0.1])
W2 = np.array([[-1.0, 1.0],
               [-1.2, 1.2],
               [0.8, -0.8]])
b2 = np.array([0.6, -0.6])

x_case = Xs[(acdc_table["patient"] == "patient134").to_numpy()][0]   # vakanın standartlaştırılmış SVDSHİ'si ve en kalın duvarı
hidden = np.maximum(0, x_case @ W1 + b1)
out = softmax(hidden @ W2 + b2)
print("x =", np.round(x_case, 3))
print("gizli katman =", np.round(hidden, 3))
print("olasılıklar (normal, anormal) =", np.round(out, 3))

**Python notu.** `np.array([[...], [...]])` bir tabloyu satır satır yazar: `W1` 2 satır (girdiler) ve 3 sütundan (nöronlar) oluşur. `x_case @ W1` her gizli nöron için bir ağırlıklı toplam verir. `softmax` içinde en büyük değerin çıkarılması sonucu değiştirmez, ancak büyük sayılarda `np.exp` fonksiyonunun taşmasını önler. Buradaki ağırlıklar ders sayfasındaki çözümlü örnekte olduğu gibi elle verilmiştir; 2. bölümde bir ağ onları kendisi öğrenir.

**Gözlemle ve düşün.** `np.maximum(0, ...)` yerine birim fonksiyonu, yani `x_case @ W1 + b1` ifadesini kullanın ve iki katmanın birlikte neye indirgendiğini kâğıt üzerinde gösterin. Bir ağ, katmanları arasında neden doğrusal olmayan bir aktivasyona ihtiyaç duyar?

**Alıştırma A.** Bir nöron iki standartlaştırılmış ölçüm (`x = [1.2, -0.5]`) almaktadır. Nöronun ağırlıkları `w = [0.8, -1.5]`, sabit terimi ise `b = 0.1` olarak verilmiştir ve aktivasyon fonksiyonu sigmoiddir. Nöronun çıktısı nedir? Sonucu üç ondalık basamağa yuvarlayın.

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma A", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
z_a = np.array([0.8, -1.5]) @ np.array([1.2, -0.5]) + 0.1
cevap = round(float(sigmoid(z_a)), 3)
teach.check("Alıştırma A", cevap)

## 2. Öğrenme: PyTorch ile kayıp, gradyanlar ve eğitim döngüsü

*Ders: [Öğrenme: Kayıp, gradyanlar ve eğitim döngüsü](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#learning-loss-gradients-and-the-training-loop)*

Bir ağ, tahminleri yanlış oldukça büyüyen bir sayıyı, yani *kaybı* azaltarak öğrenir. *Geri yayılım*, türevin zincir kuralını kayıptan başlayarak geriye doğru her ağırlığa uygular ve kaybın en hızlı arttığı yönü, yani *gradyanı* verir. *Gradyan inişi* her ağırlığı, gradyanının tersi yönünde küçük bir adım ilerletir. PyTorch gradyanları kendisi hesaplar; bu özelliğe *autograd* denir.

**Formüllerden koda.** Etiketi $y$ ve tahmin edilen olasılığı $p$ olan tek bir hasta için ikili çapraz entropi $L = -[y \ln p + (1-y) \ln(1-p)]$ ile hesaplanır: `F.binary_cross_entropy(p, y)`. Sigmoid kullanan bir nöron için gradyan $\partial L/\partial w_j = (p - y)\,x_j$'dir. $w \leftarrow w - \eta\,\partial L/\partial w$ güncellemesini, `loss.backward()` satırı `w.grad` değerini doldurduktan sonra `optimizer.step()` yapar. Birden çok sınıf olduğunda `F.cross_entropy(logits, labels)` softmax'ı ve çapraz entropiyi tek adımda uygular.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 2. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB03_derin_ogrenme_ve_evrisimli_aglar.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(2)

In [ ]:
# Autograd gradyan formülünü doğrular
x_t = torch.tensor([1.2, -0.5])
w_t = torch.tensor([0.8, -1.5], requires_grad=True)
b_t = torch.tensor(0.1, requires_grad=True)
p_t = torch.sigmoid(w_t @ x_t + b_t)
loss_t = F.binary_cross_entropy(p_t, torch.tensor(1.0))
loss_t.backward()
print("p =", round(p_t.item(), 3), " kayıp =", round(loss_t.item(), 4))
print("autograd:", np.round(w_t.grad.numpy(), 4), round(b_t.grad.item(), 4))
print("formül (p − y) · x:", np.round((p_t.item() - 1.0) * x_t.numpy(), 4))

**Python notu.** Tensör, PyTorch'taki dizinin adıdır. `requires_grad=True`, PyTorch'un tensörle yapılan her işlemi kaydetmesini sağlar; böylece `loss_t.backward()` bu işlemlerin üzerinden geriye giderek gradyanı `w_t.grad` içine yazabilir. `.item()` tek değerli bir tensörü Python sayısına, `.numpy()` ise bir tensörü NumPy dizisine çevirir.

In [ ]:
# 8 gizli nöronlu bir ağ iki ölçümden öğrenir
Xn_t = torch.tensor(Xn, dtype=torch.float32)
tn_t = torch.tensor(tn)
torch.manual_seed(RANDOM_STATE)
mlp = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 2))
optimizer = torch.optim.Adam(mlp.parameters(), lr=0.05)
for step in range(301):
    loss = F.cross_entropy(mlp(Xn_t), tn_t)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 100 == 0:
        print(step, round(loss.item(), 4))

g1, g2 = np.meshgrid(np.linspace(-2.5, 5, 200), np.linspace(-2.5, 5, 200))
grid = np.c_[g1.ravel(), g2.ravel()]
with torch.no_grad():
    p_mlp = F.softmax(mlp(torch.tensor(grid, dtype=torch.float32)), dim=1)[:, 1].numpy().reshape(g1.shape)
p_one = sigmoid(grid @ w + b).reshape(g1.shape)
groups = acdc_table["group"].to_numpy()
fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharey=True)
for ax, pr, name in [(axes[0], p_one, "tek nöron"), (axes[1], p_mlp, "8 gizli nöron")]:
    ax.contourf(g1, g2, pr, levels=[0, 0.5, 1], colors=["#EAF1F7", "#F6E0E0"])
    for g, colour in [("NOR", "#1F5F8B"), ("DCM", "#9B1B1B"), ("HCM", "#B97813")]:
        ax.scatter(Xs[groups == g, 0], Xs[groups == g, 1], s=18, color=colour, label=g)
    ax.set_title(name)
    ax.set_xlabel("lv_edvi (z)")
axes[0].set_ylabel("max_wall_mm (z)")
axes[1].legend(loc="upper right")
plt.show()
test_rows = ~train_rows
with torch.no_grad():
    pred_mlp = mlp(torch.tensor(Xs[test_rows], dtype=torch.float32)).argmax(1).numpy()
print("test doğruluğu: tek nöron", round(((sigmoid(Xs[test_rows] @ w + b) > 0.5) == target[test_rows]).mean(), 3),
      "| ağ", round((pred_mlp == target[test_rows]).mean(), 3))

**Python notu.** `nn.Sequential` katmanları art arda bağlar: `nn.Linear(2, 8)` her biri 2 girdi alan 8 nöronluk bir katmandır; ardından ReLU, ardından 2 çıktı nöronlu bir katman gelir. Her döngü Python adımındaki dört adımı içerir: Tek satırda ileri geçiş ve kayıp, eski gradyanları silmek için `optimizer.zero_grad()`, yenilerini hesaplamak için `loss.backward()` ve güncelleme için `optimizer.step()`. `torch.no_grad()`, öğrenme gerekmediğinde autograd'ın kayıt tutmasını kapatır. İki model de 60 eğitim hastasından öğrenir ve 30 test hastasında değerlendirilir.

**Gözlemle ve düşün.** Burada ventrikül dilate *ya da* duvar kalın olduğunda kalp anormal sayılır. Soldaki panelde tek bir nöron neden yalnızca düz bir sınır çizebilirken gizli katmanlı ağ, sınırını iki kardiyomiyopati grubu arasındaki köşenin çevresinden bükebilir?

**Aynı model Keras Sequential API ile.** *Ders: [Keras Sequential API ile ilk model](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#a-first-model-with-the-keras-sequential-api)*

Keras, bir önceki hücredeki ağı bir katman listesi olarak tanımlar; ardından ağı bir kayıp, bir optimizasyon yöntemi ve bir ölçütle derler ve tek çağrıyla eğitir. Hücre Keras'ı PyTorch üzerinde çalıştırır; bu yüzden TensorFlow gerekmez. Google Colab Keras'ı içerir; kişisel bir bilgisayarda paket `pip install keras` komutuyla kurulur.

In [ ]:
# Bir önceki hücredeki ağ, Keras Sequential API ile
import os

os.environ.setdefault("KERAS_BACKEND", "torch")      # Keras PyTorch üzerinde çalışır
try:
    import keras

    keras.utils.set_random_seed(RANDOM_STATE)
    model_k = keras.Sequential([
        keras.Input(shape=(2,)),                     # iki ölçüm
        keras.layers.Dense(8, activation="relu"),    # 8 nöronlu gizli katman
        keras.layers.Dense(2, activation="softmax"), # normal ya da anormal
    ])
    model_k.compile(optimizer=keras.optimizers.Adam(learning_rate=0.05),
                    loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model_k.summary()
    history_k = model_k.fit(Xn, tn, epochs=300, batch_size=len(Xn), verbose=0)
    loss_k, acc_k = model_k.evaluate(Xs[test_rows], target[test_rows], verbose=0)
    print("300 epoch sonunda eğitim kaybı", round(history_k.history["loss"][-1], 4))
    print("test doğruluğu: Keras", round(acc_k, 3), "| PyTorch", round((pred_mlp == target[test_rows]).mean(), 3))
except ImportError:
    print("keras kurulu değil; pip install keras komutuyla kurulabilir. Yukarıdaki PyTorch ağı aynı modeldir.")

**Python notu.** `os.environ.setdefault("KERAS_BACKEND", "torch")`, daha önce başka bir arka uç seçilmemişse Keras'ın PyTorch üzerinde çalışmasını sağlar; bu satır `import keras` satırından önce gelmelidir. `keras.Sequential([...])` katmanları sırayla alır. `compile` kaybı (burada 0 ve 1 olarak yazılmış etiketler için `sparse_categorical_crossentropy`), optimizasyon yöntemini ve raporlanacak ölçütü belirler. `fit`, bir önceki hücredeki eğitim döngüsünün tamamını çalıştırır: `epochs=300` geçişin her birinde 60 eğitim hastasının hepsi tek bir batch olarak kullanılır. `evaluate` test hastalarındaki kaybı ve doğruluğu döndürür; `summary()` ise katmanları 24 ve 18 parametreleriyle birlikte yazdırır.

**Alıştırma B.** Bir hastanın değerleri `x = [1.0, 2.0]`, etiketi ise `y = 1` olarak verilmiştir. Sigmoid kullanan bir nöronun ağırlıkları `w = [0.5, -0.25]`, sabit terimi ise 0'dır. İkili çapraz entropiyi PyTorch ile hesaplayın, `backward()` fonksiyonunu çağırın ve kaybın ilk ağırlığa göre gradyanını verin. Sonucu üç ondalık basamağa yuvarlayın.

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma B", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
w_b = torch.tensor([0.5, -0.25], requires_grad=True)
p_b = torch.sigmoid(w_b @ torch.tensor([1.0, 2.0]))
F.binary_cross_entropy(p_b, torch.tensor(1.0)).backward()
cevap = round(float(w_b.grad[0]), 3)
teach.check("Alıştırma B", cevap)

## 3. ACDC'nin sine MR görüntüleri: Ölçümler ve ağın girdisi

*Ders: [Dizi olarak görüntüler: ACDC'nin sine kardiyak MR görüntüleri](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#images-as-arrays-cine-cardiac-mr-of-acdc)*

ACDC'deki her inceleme, diyastol sonu (ED) ve sistol sonunda (ES) sol ventrikül kavitesinin, miyokardın ve sağ ventrikül kavitesinin uzman konturlarıyla gelir. Konturlar hacimleri, ejeksiyon fraksiyonunu (EF) ve duvarların kalınlığını verir; ACDC'nin grupları da bunlarla tanımlanır. Bu haftanın ağı ise grupları piksellerden öğrenir. Bu bölümde ağın girdisi hazırlanır: Her kesitin diyastol sonu ve sistol sonu kareleri iki kanal olarak düzenlenir ve hastalar eğitim, doğrulama ve test kümelerine ayrılır.

**Formüllerden koda.** Bir hacim, konturun içindeki voksel sayısı ile bir vokselin hacminin çarpımıdır, $V = N_{vox}\,\Delta x\,\Delta y\,\Delta z$: mL cinsinden `(gt == 3).sum() * np.prod(spacing) / 1000`. EF, $(EDV - ESV)/EDV \times 100$ formülüyle hesaplanır. Normalleştirme $x' = (x - \mu_{train})/\sigma_{train}$ yalnızca eğitim kesitlerinin ortalamasını ve SS'sini kullanır. PyTorch görüntüleri (sayı, kanal, satır, sütun) boyutunda bekler; burada bu boyut (n, 2, 64, 64)'tür.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 3. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB03_derin_ogrenme_ve_evrisimli_aglar.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(3)

In [ ]:
# Uzman konturlarında ölçülen üç gruptaki 90 hasta
print(acdc_table.groupby("group")[["lv_ef", "lv_edvi", "max_wall_mm", "rv_ef"]].agg(["mean", "std"]).round(1))
print(acdc_table.groupby(["split", "group"]).size().unstack())

# 134 numaralı hastanın ejeksiyon fraksiyonu konturlarının voksellerinden
gt_ed, spacing = read_nifti(case_dir / f"patient134_frame{int(info['ED']):02d}_gt.nii.gz")
gt_es, _ = read_nifti(case_dir / f"patient134_frame{int(info['ES']):02d}_gt.nii.gz")
voxel_ml = float(np.prod(spacing)) / 1000
edv, esv = (gt_ed == 3).sum() * voxel_ml, (gt_es == 3).sum() * voxel_ml
print("patient134: EDV", round(edv, 1), "mL · ESV", round(esv, 1), "mL · EF", round(100 * (edv - esv) / edv, 1), "%")

In [ ]:
# Normal kalp, DCM ve HCM: diyastol sonunda ortadaki kesit ve konturlar
fig, axes = plt.subplots(1, 3, figsize=(9.5, 3.4))
for ax, (pid, g) in zip(axes, [("patient139", "NOR"), ("patient113", "DCM"), ("patient138", "HCM")]):
    rows = np.where((S["patient"] == pid) & (S["phase"] == "ED"))[0]
    i = rows[len(rows) // 2]
    ax.imshow(S["images"][i], cmap="gray", vmin=0, vmax=1)
    ax.contour(S["masks"][i] == 2, levels=[0.5], colors="#E8C020", linewidths=1)
    ax.contour(S["masks"][i] == 3, levels=[0.5], colors="#FF3030", linewidths=1.4)
    r = acdc_table.set_index("patient").loc[pid]
    ax.set_title(f"{pid} ({g})\nEF {r.lv_ef:.0f} % · SVDSHİ {r.lv_edvi:.0f} · duvar {r.max_wall_mm:.1f} mm", fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

**Python notu.** `(gt_ed == 3)`, True ve False değerlerinden oluşan bir dizidir; `.sum()` True değerlerini sayar; burada bu değerler sol ventrikül kavitesinin vokselleridir. `ax.contour(mask, levels=[0.5])`, 0 ve 1'lerden oluşan bir maskenin sınırını çizer. `acdc_table.set_index("patient").loc[pid]` tek bir hastanın satırını bulur; `f"{r.lv_ef:.0f}"` bir sayıyı ondalıksız yazar. Görüntüler ve konturlar ACDC'den gelmektedir (CC BY-NC-SA 4.0).

In [ ]:
# 90 hastanın ölçümlerine ACDC kuralları
def acdc_rule(row):
    if row["lv_ef"] < 40 and row["lv_edvi"] > 100:
        return "DCM"
    if row["max_wall_mm"] > 15 and row["lv_ef"] >= 50:
        return "HCM"
    return "NOR"


acdc_table["rule"] = acdc_table.apply(acdc_rule, axis=1)
print(pd.crosstab(acdc_table["group"], acdc_table["rule"]))
print("kuralların doğruluğu:", round(float((acdc_table["rule"] == acdc_table["group"]).mean()), 3))
print(acdc_table.loc[acdc_table["rule"] != acdc_table["group"], ["patient", "group", "rule", "lv_ef", "lv_edvi", "max_wall_mm"]])

In [ ]:
# Ağın girdisi: iki kanal olarak ED ve ES, hastaya göre ayrılmış
P = acdc.pairs(GROUPS)
X, y, patients, sets = P["X"], P["y"], P["patient"], P["set"]
tr, va, te = sets == "train", sets == "val", sets == "test"
mean, sd = float(X[tr].mean()), float(X[tr].std())


def to_tensor(rows):
    return torch.tensor((X[rows] - mean) / sd, dtype=torch.float32)


Xtr, Xva, Xte = to_tensor(tr), to_tensor(va), to_tensor(te)
ytr, yva, yte = (torch.tensor(y[m], dtype=torch.long) for m in (tr, va, te))
for name, m in [("train", tr), ("val", va), ("test", te)]:
    print(name, "kesit", int(m.sum()), "hasta", len(set(patients[m])), np.bincount(y[m]))
print("eğitim piksellerinin ortalaması ve SS'si", round(mean, 4), round(sd, 4), tuple(Xtr.shape))

**Python notu.** `acdc.pairs`, her hastanın ortadaki üç kesitini (0. kanalda diyastol sonu, 1. kanalda sistol sonu karesi) ve her kesitin hangi kümede olduğunu döndürür. ACDC'nin 30 test hastası test kümesini, ACDC eğitim kümesindeki her gruptan 5 hasta da doğrulama kümesini oluşturur. Hiçbir hasta iki kümede birden yer almaz. `np.bincount(y[m])`, 0, 1 ve 2. sınıfların kesitlerini sayar. Solunda üç ad bulunan satır, bir *üreteç ifadesinin*, yani parantez içinde yazılmış bir döngünün ürettiği üç tensörü bu adlara dağıtır.

**Alıştırma C.** Test kümesinde kaç kesit vardır?

In [ ]:
veri = alistirma_tablosu("C")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma C", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = int(te.sum())
teach.check("Alıştırma C", cevap)

## 4. Elle ve PyTorch ile evrişim

*Ders: [Konvolüsyon işlemi: Görüntüde kenar/desen bulma](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#convolution-filters-that-slide-over-an-image)*

Bir evrişim, küçük bir ağırlık tablosunu, yani *çekirdeği*, görüntü üzerinde kaydırır. Her konumda çekirdeği altındaki piksellerle çarpar ve çarpımları toplar; bu, çıktının, yani *öznitelik haritasının*, bir pikselini verir. Aynı çekirdek her yerde kullanıldığı için bir katman az ağırlıkla yetinir ve bir örüntü nerede görünürse görünsün bulunur. *Maksimum havuzlama* her 2 × 2'lik bloğun en büyük değerini tutar ve boyutu yarıya indirir.

**Formüllerden koda.** Çıktı $Y(i, j) = \sum_{u}\sum_{v} K(u, v)\,X(i+u, j+v) + b$ ile hesaplanır: `F.conv2d(x, kernel, bias)`. Girdi $n$ piksel, çekirdek $k$, dolgu $p$ ve adım $s$ olduğunda çıktı $\lfloor (n + 2p - k)/s \rfloor + 1$ pikseldir: `conv_out(n, k, s, p)`. Bu formül aslında bir çapraz korelasyondur; ağ çekirdeği her iki durumda da öğrendiği için derin öğrenmede evrişim adı kullanılır.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 4. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB03_derin_ogrenme_ve_evrisimli_aglar.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(4)

In [ ]:
# 5 × 5'lik bir yamada 3 × 3'lük bir çekirdek, elle ve PyTorch ile
patch = np.array([[0, 0, 0, 0, 0],
                  [0, 1, 1, 1, 0],
                  [0, 1, 0, 1, 0],
                  [0, 1, 1, 1, 0],
                  [0, 0, 0, 0, 0]], dtype=float)
kernel = np.array([[-1, 0, 1],
                   [-1, 0, 1],
                   [-1, 0, 1]], dtype=float)          # dikey kenarlar
by_hand = np.zeros((3, 3))
for r in range(3):
    for c in range(3):
        by_hand[r, c] = np.sum(patch[r:r + 3, c:c + 3] * kernel)
print(by_hand)
with_torch = F.conv2d(torch.tensor(patch)[None, None], torch.tensor(kernel)[None, None])
print(with_torch[0, 0].numpy())


def conv_out(n, k, s=1, p=0):
    return (n + 2 * p - k) // s + 1


for n, k, s, p in [(64, 3, 1, 1), (64, 3, 1, 0), (64, 5, 2, 2), (32, 2, 2, 0)]:
    print(f"n={n} k={k} s={s} p={p} -> {conv_out(n, k, s, p)}")

**Python notu.** İç içe iki `for` döngüsü 3 × 3'lük çıktının her konumunu dolaşır; `patch[r:r + 3, c:c + 3]` o konumdaki 3 × 3'lük penceredir. `[None, None]` öne iki boyut ekler, çünkü `F.conv2d` görüntü için (sayı, kanal, satır, sütun), çekirdek için (çıkış kanalı, giriş kanalı, satır, sütun) boyutunu bekler.

In [ ]:
# DCM'li bir hasta olan 113 numaralı hastanın diyastol sonu kesitinde üç çekirdek
rows = np.where((S["patient"] == "patient113") & (S["phase"] == "ED"))[0]
slice_d = S["images"][rows[len(rows) // 2]]
kernels = {"dikey kenarlar": [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]],
           "yatay kenarlar": [[-1, -2, -1], [0, 0, 0], [1, 2, 1]],
           "bulanıklaştırma": [[1 / 9, 1 / 9, 1 / 9]] * 3}
t = torch.tensor(slice_d)[None, None]
fig, axes = plt.subplots(1, 4, figsize=(11, 3))
axes[0].imshow(slice_d, cmap="gray")
axes[0].set_title("patient113 (DCM)")
for ax, (name, k) in zip(axes[1:], kernels.items()):
    fmap = F.conv2d(t, torch.tensor(k, dtype=torch.float32)[None, None], padding=1)[0, 0]
    ax.imshow(fmap.numpy(), cmap="RdBu_r")
    ax.set_title(name)
for ax in axes:
    ax.axis("off")
plt.show()
print(tuple(t.shape), "→ max pooling →", tuple(F.max_pool2d(t, 2).shape))

**Python notu.** Bir sözlük, adları ve değerleri çiftler hâlinde tutar; `.items()` bu çiftleri döngü için tek tek döndürür. `padding=1` görüntünün çevresine sıfırlardan bir çerçeve ekler; böylece 3 × 3'lük bir çekirdek boyutu 64 × 64'te tutar. `RdBu_r` renk haritası negatif değerleri mavi, pozitif değerleri kırmızı gösterir; bir kenar filtresi bir duvarın iki yanında böyle cevap verir, çünkü görüntü parlak kaviteden koyu miyokarda doğru kararır.

**Elle havuzlama.** *Ders: [Pooling, Dropout, Batch Normalization](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#pooling-dropout-batch-normalization)*

Hücre, dersteki çözümlü örneğin 4 × 4'lük öznitelik haritasını 2 × 2'lik bloklar üzerinde maksimum ve ortalama havuzlamayla küçültür.

In [ ]:
# 4 × 4'lük bir öznitelik haritasında maksimum ve ortalama havuzlama
fmap4 = torch.tensor([[1., 3., 0., 2.],
                      [4., 2., 1., 0.],
                      [0., 1., 5., 6.],
                      [2., 0., 3., 1.]])[None, None]
print("maksimum havuzlama:\n", F.max_pool2d(fmap4, 2)[0, 0])
print("ortalama havuzlama:\n", F.avg_pool2d(fmap4, 2)[0, 0])

**Python notu.** `[None, None]` başa iki boyut ekler: Tek kanallı tek bir görüntüden oluşan bir batch. Bunun nedeni PyTorch'un havuzlama fonksiyonlarının (batch, kanal, satır, sütun) biçiminde bir dizi beklemesidir. `F.max_pool2d(x, 2)` ve `F.avg_pool2d(x, 2)` örtüşmeyen 2 × 2'lik bloklar alır; `[0, 0]` ise tek haritayı yeniden dışarı çıkarır.

**Alıştırma D.** 5 × 5 çekirdekli, adımı 2 ve dolgusu 1 olan bir evrişim 64 × 64 piksellik bir girdi üzerinde çalışmaktadır. Çıktının bir kenarının boyutu nedir?

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma D", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = conv_out(64, 5, 2, 1)
print(F.conv2d(torch.zeros(1, 1, 64, 64), torch.zeros(1, 1, 5, 5), stride=2, padding=1).shape)
teach.check("Alıştırma D", cevap)

## 5. İlk evrişimli ağ

*Ders: [Sine MR için bir evrişimli ağ](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#a-convolutional-network-for-cine-mr)*

Dersin ağı üç bloktan oluşur. Her blokta 3 × 3 çekirdekli bir evrişim, batch normalizasyonu, ReLU ve maksimum havuzlama bulunur. Görüntü 64 pikselden 32, 16 ve 8 piksele küçülürken öznitelik haritalarının sayısı ED ve ES kanallarından oluşan 2'den 8, 16 ve 32'ye çıkar. Bir doğrusal katman 32 × 8 × 8 = 2048 değeri her grup için bir tane olmak üzere üç çıktıya çevirir. 2. bölümdeki eğitim döngüsü artık 16 kesitlik *mini-batch'lerle* çalışır; bütün eğitim kesitlerinin ağdan bir kez geçirilmesine *epoch* denir. Her epoch'un sonunda doğrulama kesitlerindeki kayıp, hangi ağırlıkların saklanacağını belirler.

**Formüllerden koda.** `CmrCNN` sınıfı katmanları `__init__` içinde sıralar ve bir görüntünün bunlardan nasıl geçeceğini `forward` içinde tanımlar. `train_model` epoch'ları çalıştırır ve her epoch'un eğitim ve doğrulama kaybını döndürür. `F.softmax(model(x), dim=1)` üç grubun olasılıklarını verir; bir hasta için karar, hastanın kesitleri üzerinden ortalama olasılığı en yüksek olan gruptur.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 5. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB03_derin_ogrenme_ve_evrisimli_aglar.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(5)

In [ ]:
class CmrCNN(nn.Module):
    """Üç evrişim bloğu ve bir doğrusal katman; girdi: bir kesitin iki karesi, iki kanal olarak."""

    def __init__(self, n_classes=3, dropout=0.3, batchnorm=True, in_channels=2):
        super().__init__()
        layers, c_in = [], in_channels
        for c_out in (8, 16, 32):
            layers += [nn.Conv2d(c_in, c_out, 3, padding=1)]
            layers += [nn.BatchNorm2d(c_out)] if batchnorm else []
            layers += [nn.ReLU(), nn.MaxPool2d(2)]
            c_in = c_out
        self.features = nn.Sequential(*layers)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(32 * 8 * 8, n_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


print(CmrCNN())
with torch.no_grad():
    print(CmrCNN()(Xtr[:4]).shape)

**Python notu.** Bir *sınıf* nesneler için bir tariftir. `class CmrCNN(nn.Module)` PyTorch'un `nn.Module` sınıfı üzerine kurulur; `super().__init__()` üst sınıfın önce kendini hazırlamasını sağlar. `self.features` katmanları nesnenin içinde saklar. `layers += [...]` bir listeye ekleme yapar; `[...] if batchnorm else []` batch normalizasyonunu yalnızca istendiğinde ekler. `nn.Sequential(*layers)` listeyi ayrı argümanlara açar.

**Alıştırma E.** `CmrCNN()` modelinin kaç eğitilebilir parametresi vardır?

In [ ]:
veri = alistirma_tablosu("E")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma E", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = sum(p.numel() for p in CmrCNN().parameters() if p.requires_grad)
teach.check("Alıştırma E", cevap)

In [ ]:
def train_model(model, X_tr, y_tr, X_va, y_va, epochs=40, lr=0.003, batch=16, seed=RANDOM_STATE, augment_fn=None):
    torch.manual_seed(seed)
    order_gen = torch.Generator().manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    best, best_state, history = float("inf"), None, []
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(y_tr), generator=order_gen)
        total = 0.0
        for start in range(0, len(y_tr), batch):
            j = order[start:start + batch]
            xb = X_tr[j] if augment_fn is None else augment_fn(X_tr[j], order_gen)
            loss = F.cross_entropy(model(xb), y_tr[j])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(j)
        model.eval()
        with torch.no_grad():
            val_loss = F.cross_entropy(model(X_va), y_va).item()
        history.append((total / len(y_tr), val_loss))
        if val_loss < best:
            best, best_state = val_loss, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return history


def by_patient(model, X_t, rows):
    """Her hastanın kesitlerinin ortalama olasılıkları, gerçek sınıflar ve kararlar."""
    model.eval()
    with torch.no_grad():
        prob = F.softmax(model(X_t), dim=1).numpy()
    names = sorted(set(patients[rows]))
    mean_p = np.array([prob[patients[rows] == n].mean(axis=0) for n in names])
    true = np.array([y[rows][patients[rows] == n][0] for n in names])
    return names, mean_p, true, mean_p.argmax(axis=1)

**Python notu.** `model.train()` ve `model.eval()` dropout'u ve batch normalizasyonunu eğitim ve değerlendirme davranışları arasında değiştirir. `torch.randperm` her epoch'ta kesitlerin sırasını karıştırır. `copy.deepcopy(model.state_dict())` ağırlıkların tam bir kopyasını saklar; `load_state_dict` de sonunda en iyi kopyayı geri yükler. Her batch'in kaybı, batch'in büyüklüğüyle (`len(j)`) ağırlıklandırılır; böylece daha küçük olan son batch doğru oranda hesaba katılır. `by_patient`, her hastanın üç kesitinin olasılıklarının ortalamasını alır.

In [ ]:
# Erken durdurmayla 40 epoch eğitim, yaklaşık bir dakika
torch.manual_seed(RANDOM_STATE)
model = CmrCNN()
history = train_model(model, Xtr, ytr, Xva, yva, epochs=40)
best_epoch = int(np.argmin([h[1] for h in history])) + 1
names_te, prob_te, true_te, pred_te = by_patient(model, Xte, te)
print("en iyi epoch", best_epoch, "· test hastası", len(names_te),
      "· doğruluk", round(float((pred_te == true_te).mean()), 3))

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(range(1, len(history) + 1), [h[0] for h in history], color="#1F5F8B", label="eğitim")
ax.plot(range(1, len(history) + 1), [h[1] for h in history], color="#9B1B1B", label="doğrulama")
ax.axvline(best_epoch, color="#888888", ls="--", lw=1)
ax.set_xlabel("epoch")
ax.set_ylabel("çapraz entropi kaybı")
ax.legend()
plt.show()

**Gözlemle ve düşün.** Dersin çalıştırmasında aynı ağ aynı seed değeriyle eğitilmiş, en iyi epoch 37 olmuş ve 30 test hastasının 28'i doğru sınıflanmıştır; yani doğruluk 0,933'tür. Siz de aynı sayıları elde ediyor musunuz? Bilgisayarlar arasındaki küçük farklar, kayan noktalı işlemlerin sırasından kaynaklanır. Burada bir hastanın fazlası ya da eksiği doğruluğu neden 0,033 değiştirir ve bu durum, 30 hastadaki sonuçların kesinliği hakkında ne söyler?

## 6. Aşırı öğrenme: Dropout, batch normalizasyonu ve veri çoğaltma

*Ders: [Overfitting tespiti ve önleme](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#overfitting-dropout-batch-normalisation-and-augmentation)*

On iki bin ağırlıklı bir ağ, 135 eğitim kesitini ezberleyebilir. Bu durumda eğitim kaybı düşmeyi sürdürürken doğrulama kaybı düşmeyi bırakır ya da yükselir. Dropout, batch normalizasyonu, veri çoğaltma ve erken durdurma, aşırı öğrenmeye karşı kullanılır.

**Formüllerden koda.** Dropout, eğitim sırasında bir birimi $1 - p$ olasılıkla korur ve korunan birimin değerini $1 - p$'ye böler: $\tilde{a} = m\,a/(1 - p)$; kodda bu işlemi `nn.Dropout(p)` yapar. Batch normalizasyonu mini-batch üzerinden $\hat{x} = (x - \mu_B)/\sqrt{\sigma_B^2 + \varepsilon}$ değerini, ardından $\gamma\hat{x} + \beta$ değerini hesaplar; kodda `nn.BatchNorm2d(c)` kullanılır. Eğitimde veri çoğaltma için bir kesit 0, 90, 180 ya da 270 derece döndürülür (`torch.rot90`): Döndürme anatomiyi korur, yalnızca bakış yönünü değiştirir. Ayna görüntüsü ise, nadir görülen ayna görüntüsü dekstrokardisinde olduğu gibi, kalbin sağ ve sol tarafının yerini değiştirirdi.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 6. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB03_derin_ogrenme_ve_evrisimli_aglar.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(6)

In [ ]:
# Veri çoğaltma: 134 numaralı hastanın bir kesiti çeyrek tur adımlarla döndürülüyor
def augment(xb, gen):
    k = int(torch.randint(0, 4, (1,), generator=gen))
    return torch.rot90(xb, k, dims=(2, 3))


rows = np.where((S["patient"] == "patient134") & (S["phase"] == "ED"))[0]
case_slice = torch.tensor(S["images"][rows[1]])
fig, axes = plt.subplots(1, 4, figsize=(9, 2.6))
for k, ax in enumerate(axes):
    ax.imshow(torch.rot90(case_slice, k).numpy(), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{90 * k}°")
    ax.axis("off")
plt.show()

**Python notu.** `torch.rot90(x, k, dims=(2, 3))` bir batch'teki görüntüleri satır ve sütunlarının düzleminde k çeyrek tur döndürür. `torch.randint(0, 4, (1,), generator=gen)` kendi rastgele üreteciyle 0 ile 3 arasında tek bir tam sayı çeker; böylece aynı seed değeri aynı veri çoğaltmayı yeniden verir. Her mini-batch bir kez döndürülür; ağ birçok epoch boyunca her kesiti dört yönelimin hepsinde görür.

**Kardiyak görüntüler için görüntü augmentation.** *Ders: [Görüntü augmentation kardiyak görüntüler için](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#image-augmentation-for-cardiac-images)*

Döndürmeler, veri çoğaltmanın ailelerinden biridir. Hücre iki yöntem daha, yani düşük kontrastı ve küçük bir kaydırmayı, ayrıca kalbin sağ ve sol tarafının yerini değiştireceği için eğitimde kullanılmayan ayna görüntüsünü gösterir. Aşağıdaki eğitim yalnızca döndürmeleri kullanır.

In [ ]:
# İki veri çoğaltma yöntemi daha ve eğitimin kullanmadığı ayna görüntüsü
lower_contrast = 0.5 + 0.6 * (case_slice - 0.5)                  # kontrast yüzde 40 düşürüldü
shifted = torch.roll(case_slice, shifts=(0, 6), dims=(0, 1))     # altı piksel sağa
mirror = torch.flip(case_slice, dims=(1,))                        # kullanılmaz: kalbin tarafları yer değiştirir
fig, axes = plt.subplots(1, 4, figsize=(9, 2.6))
for ax, (img, title) in zip(axes, [(case_slice, "özgün"), (lower_contrast, "düşük kontrast"),
                                   (shifted, "6 piksel kaydırma"), (mirror, "ayna, kullanılmaz")]):
    ax.imshow(img.numpy(), cmap="gray", vmin=0, vmax=1)
    ax.set_title(title, fontsize=9)
    ax.axis("off")
plt.show()

**Python notu.** `0.5 + 0.6 * (x - 0.5)` her pikseli orta griye doğru çeker ve kontrastı yüzde 40 düşürür. `torch.roll` görüntüyü altı sütun kaydırır; sağdan çıkan pikseller soldan geri girer. Bu durum, ortalanmış bir kalp çevresindeki küçük bir kaydırma için kabul edilebilir. `torch.flip(x, dims=(1,))` sütunların sırasını tersine çevirir ve ayna görüntüsünü verir.

In [ ]:
# Düzenlileştirmesiz ağ ile batch normalizasyonlu ve dropout'lu ağ; her biri 60 epoch, yaklaşık bir dakika
curves = {}
for name, kw in [("düzenlileştirme yok", dict(dropout=0.0, batchnorm=False)),
                 ("dropout 0,5 ve batch normalizasyonu", dict(dropout=0.5, batchnorm=True))]:
    torch.manual_seed(RANDOM_STATE)
    net = CmrCNN(**kw)
    curves[name] = train_model(net, Xtr, ytr, Xva, yva, epochs=60)
    _, _, true_n, pred_n = by_patient(net, Xte, te)
    best = int(np.argmin([h[1] for h in curves[name]])) + 1
    print(name, "· en iyi epoch", best, "· test doğruluğu", round(float((pred_n == true_n).mean()), 3))
fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4), sharey=True)
for ax, (name, h) in zip(axes, curves.items()):
    ax.plot(range(1, 61), [v[0] for v in h], color="#1F5F8B", label="eğitim")
    ax.plot(range(1, 61), [v[1] for v in h], color="#9B1B1B", label="doğrulama")
    ax.axvline(int(np.argmin([v[1] for v in h])) + 1, color="#888888", ls="--", lw=1)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel("epoch")
axes[0].set_ylabel("kayıp")
axes[0].legend()
plt.show()

**Gözlemle ve düşün.** Düzenlileştirme olmadan eğitim kaybı neredeyse 0'a inerken doğrulama kaybı birkaç epoch sonra yükselir: Ağ 135 kesiti ezberlemektedir. Erken durdurma her panelde hangi epoch'u saklardı? `epochs=60` değerini 120 yapın ve hücreyi yeniden çalıştırın: İki eğri arasındaki açıklığa ve test doğruluğuna ne olur?

In [ ]:
# Küçük tensörlerde dropout ve batch normalizasyonu
drop = nn.Dropout(p=0.5)
drop.train()
torch.manual_seed(1)
print("eğitim modunda dropout:", drop(torch.ones(8)).numpy())
drop.eval()
print("değerlendirme modunda dropout:", drop(torch.ones(8)).numpy())
bn = nn.BatchNorm1d(1)
values = torch.tensor([[9.0], [10.0], [12.0], [13.0]])
print("batch normalizasyonu:", bn(values).detach().numpy().ravel().round(3))

**Alıştırma F.** Eğitim modunda p = 0,3 olan dropout, değerlerin ortalama yüzde 30'unu 0 yapar. `torch.ones(10)` dizisinin korunan elemanları hangi değeri alır? Sonucu üç ondalık basamağa yuvarlayın.

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma F", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
d03 = nn.Dropout(p=0.3)
d03.train()
out03 = d03(torch.ones(10))
cevap = round(float(out03[out03 > 0][0]), 3)
teach.check("Alıştırma F", cevap)

## 7. Sonucu okumak: Kesinlik, duyarlılık ve F1

*Ders: [Confusion matrix, F1-score yorumlama](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#reading-the-result-precision-recall-and-f1)*

Bir ağın doğruluğu, kaç hastayı doğru sınıfladığını söyler; hangilerini kaçırdığını söylemez. Her sınıf için *kesinlik*, ağın o sınıfa yerleştirdiği hastalar arasındaki doğru kararların oranıdır; *duyarlılık* ise sınıfa ait hastalardan bulunanların oranıdır. *F1 skoru* ikisini birleştirir; *makro F1* ise sınıfların F1 değerlerinin düz ortalamasıdır.

**Formüllerden koda.** Satırlarında gerçek sınıfların, sütunlarında kararların bulunduğu bir karışıklık matrisinde $k$ sınıfının $TP$ değeri köşegen elemanı `cm[k, k]`, $FP$ değeri sütununun geri kalanı, $FN$ değeri ise satırının geri kalanıdır. $P = TP/(TP + FP)$, $R = TP/(TP + FN)$ ve $F1 = 2\,TP/(2\,TP + FP + FN)$ olarak hesaplanır; `f1_score(true, pred, average="macro")` ise makro F1'i verir.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 7. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB03_derin_ogrenme_ve_evrisimli_aglar.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(7)

In [ ]:
# 5. bölümdeki ağ 30 test hastasında, NumPy ile elle
cm = confusion_matrix(true_te, pred_te, labels=[0, 1, 2])
print(pd.DataFrame(cm, index=["NOR", "DCM", "HCM"], columns=["NOR", "DCM", "HCM"]))
tp = np.diag(cm)
fp = cm.sum(axis=0) - tp
fn = cm.sum(axis=1) - tp
precision = tp / np.maximum(tp + fp, 1)
recall = tp / np.maximum(tp + fn, 1)
f1 = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
print(pd.DataFrame({"precision": precision, "recall": recall, "F1": f1}, index=["NOR", "DCM", "HCM"]).round(3))
print("makro F1", round(float(f1.mean()), 3), "· sklearn", round(f1_score(true_te, pred_te, average="macro"), 3))

**Python notu.** `np.diag(cm)` matrisin köşegenini, yani her sınıfın doğru kararlarını alır. `cm.sum(axis=0)` her sütunu, `cm.sum(axis=1)` her satırı toplar. `np.maximum(..., 1)`, bir sınıfa hiç karar verilmediğinde sıfıra bölmeyi önler. Elle hesaplama ile `f1_score` aynı sonucu vermelidir.

In [ ]:
# Yanlış sınıflanan hastalar ve ölçümleri
classes = ["NOR", "DCM", "HCM"]
errors = [(n, classes[t_], classes[p_], np.round(pr, 3)) for n, t_, p_, pr in zip(names_te, true_te, pred_te, prob_te) if t_ != p_]
measures = acdc_table.set_index("patient")
for n, t_, p_, pr in errors:
    r = measures.loc[n]
    print(n, "gerçek", t_, "· ağ", p_, pr, "· EF", r.lv_ef, "· en kalın duvar", r.max_wall_mm, "mm")

**Gözlemle ve düşün.** Dersin çalıştırmasında ağ HCM'li iki hastayı kaçırır ve onlara normal der; 134 numaralı hasta bunlardan biridir. İkisinin de en kalın segmenti 15 mm'nin üzerindedir; bu yüzden ölçümlere uygulanan ACDC kuralları onları doğru gruba yerleştirir. Ağ yalnızca ortadaki üç kesiti görür. En kalın segment kalbin neresinde olabilir ve bunu nasıl denetlersiniz?

In [ ]:
# 134 numaralı hasta: diyastol sonunda her kesitteki en kalın duvar
walls = acdc.wall_by_slice(gt_ed, spacing)
lv_slices = [k for k in range(gt_ed.shape[2]) if (gt_ed[:, :, k] == 3).any()]
mid_k = lv_slices[len(lv_slices) // 2]
seen = sorted(sorted(lv_slices, key=lambda k: (abs(k - mid_k), k))[:3])
print("kesit, en kalın duvar (mm), tabandan apekse")
for k in lv_slices:
    print(k, round(walls[k], 1), "ağın gördüğü kesit" if k in seen else "")

**Python notu.** `acdc.wall_by_slice` bir sözlük döndürür; sözlükte her kesitin numarası ve o kesitteki duvarın en kalın noktasının mm cinsinden değeri bulunur. Fonksiyon, her kesitte endokard sınırının her noktasından duvarın dışındaki (epikard tarafındaki ya da sağ ventriküldeki) en yakın noktaya olan uzaklığı ölçer ve en büyüğünü saklar. Fonksiyonun içinde `ndimage.distance_transform_edt`, her piksele, kavitenin ve duvarın dışındaki en yakın piksele olan uzaklığını verir. ACDC'nin kesitleri tabandan (0. kesit) apekse doğru numaralanır.

**Alıştırma G.** Bir ağ NOR, DCM ve HCM için `cm3 = [[10, 0, 0], [0, 10, 0], [2, 0, 8]]` karışıklık matrisini vermektedir (satırlarda gerçek sınıflar yer alır). Makro F1 değeri nedir? Sonucu üç ondalık basamağa yuvarlayın.

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma G", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cm3 = np.array([[10, 0, 0], [0, 10, 0], [2, 0, 8]])
tp3 = np.diag(cm3)
cevap = round(float(np.mean(2 * tp3 / (2 * tp3 + (cm3.sum(axis=0) - tp3) + (cm3.sum(axis=1) - tp3)))), 3)
teach.check("Alıştırma G", cevap)

## 8. Bilinen mimariler ve atlama bağlantıları

*Ders: [VGG16, ResNet50 mimarileri (kavramsal)](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#known-architectures-from-lenet-to-resnet)*

Evrişimli ağlar, el yazısı rakamları yaklaşık altmış bin ağırlıkla okuyan LeNet-5'ten milyonlarca ağırlığı ve yüzden fazla katmanı olan ağlara kadar büyümüştür. Çok derin ağlar ResNet'in *atlama bağlantısıyla* eğitilebilir hâle gelmiştir: Bir blok girdisini çıktısına ekler ($y = F(x) + x$); böylece gradyan bloğun yanından akabilir ve ekleyecek bir şeyi olmayan bir blok $F(x) = 0$ öğrenebilir.

**Formüllerden koda.** $c_{in}$ giriş kanalı, $c_{out}$ çıkış kanalı ve $k \times k$ çekirdeği olan bir evrişim katmanında $c_{in} \cdot c_{out} \cdot k^2 + c_{out}$ parametre bulunur; son terim sabit terimler içindir. Bir batch normalizasyonu katmanında $2c$ parametre vardır: $c$ kanalın her biri için $\gamma$ ve $\beta$. `count(model)` eğitilebilir bütün tensörlerin boyutlarını toplar.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 8. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB03_derin_ogrenme_ve_evrisimli_aglar.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(8)

In [ ]:
# LeNet-5'in, bir VGG bloğunun ve bir artık bloğun parametreleri
def count(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


lenet5 = nn.Sequential(nn.Conv2d(1, 6, 5), nn.Tanh(), nn.AvgPool2d(2),
                       nn.Conv2d(6, 16, 5), nn.Tanh(), nn.AvgPool2d(2), nn.Flatten(),
                       nn.Linear(16 * 5 * 5, 120), nn.Tanh(), nn.Linear(120, 84), nn.Tanh(), nn.Linear(84, 10))
vgg_block = nn.Sequential(nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(),
                          nn.Conv2d(128, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))


class BasicBlock(nn.Module):
    """ResNet'in temel bloğu: iki 3 × 3 evrişim ve bir atlama bağlantısı."""

    def __init__(self, c):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(c)
        self.conv2 = nn.Conv2d(c, c, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(c)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.relu(out + x)            # atlama bağlantısı


print("LeNet-5 (32 × 32):", count(lenet5), "|", tuple(lenet5(torch.zeros(1, 1, 32, 32)).shape))
print("VGG bloğu 64 → 128:", count(vgg_block))
print("CmrCNN:", count(CmrCNN()))
try:
    import torchvision
    print("ResNet-18 (torchvision):", count(torchvision.models.resnet18(weights=None)))
    print("ResNet-50 (torchvision):", count(torchvision.models.resnet50(weights=None)))
except ImportError:
    print("torchvision kurulu değil; ResNet-18'in 11.689.512, ResNet-50'nin 25.557.032 parametresi vardır.")

**Python notu.** `bias=False` evrişimlerin sabit terimlerini çıkarır, çünkü hemen ardından gelen batch normalizasyonunun kendi kaydırma terimi $\beta$ vardır. `forward` içindeki `out + x` atlama bağlantısıdır; `out` ile `x`'in aynı boyutta olmasını gerektirir ve 1'lik dolgu bunu sağlar. `weights=None`, torchvision ağlarını rastgele ağırlıklarla kurar; böylece hiçbir şey indirilmez. 4. hafta ImageNet üzerinde öğrenilmiş ağırlıkları yükler.

In [ ]:
# VGG-16 ve ResNet-50'nin darboğaz bloğu
plain = nn.Sequential(nn.Conv2d(256, 256, 3, padding=1, bias=False), nn.Conv2d(256, 256, 3, padding=1, bias=False))
bottleneck = nn.Sequential(nn.Conv2d(256, 64, 1, bias=False),
                           nn.Conv2d(64, 64, 3, padding=1, bias=False),
                           nn.Conv2d(64, 256, 1, bias=False))
print("256 kanal üzerinde iki 3 × 3 evrişim:", count(plain))
print("darboğaz 1 × 1, 3 × 3, 1 × 1:", count(bottleneck),
      "| oran", round(count(bottleneck) / count(plain), 3))
try:
    import torchvision
    vgg16 = torchvision.models.vgg16(weights=None)
    print("VGG-16 (torchvision):", count(vgg16), "| üç tam bağlantılı katmanda:", count(vgg16.classifier),
          "| oran", round(count(vgg16.classifier) / count(vgg16), 3))
except ImportError:
    print("torchvision kurulu değil; VGG-16'nın 138.357.544 parametresi vardır ve bunların 123.642.856'sı tam bağlantılı katmanlardadır.")

**Python notu.** 1 × 1'lik bir evrişim (`nn.Conv2d(256, 64, 1)`), her pikseldeki 256 kanalı komşulara bakmadan 64 yeni kanalda karıştırır; böylece kanal sayısını ucuza değiştirir. `vgg16.classifier`, VGG-16'nın evrişimlerden sonraki kısmı, yani üç tam bağlantılı katmanıdır; `count` bu katmanların parametrelerini toplar.

**Alıştırma H.** 16 giriş kanalı, 32 çıkış kanalı, 3 × 3 çekirdeği ve sabit terimleri olan bir evrişim katmanının kaç parametresi vardır?

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma H", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = count(nn.Conv2d(16, 32, 3))
teach.check("Alıştırma H", cevap)

**Alıştırma I.** `BasicBlock(64)` bloğunun kaç eğitilebilir parametresi vardır?

In [ ]:
cevap = None   # cevabınız
teach.check("Alıştırma I", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
cevap = count(BasicBlock(64))
print(BasicBlock(64)(torch.randn(1, 64, 16, 16)).shape)
teach.check("Alıştırma I", cevap)

## 9. YZ asistanıyla çalışmak, 3. bölüm: Bir eğitim döngüsünü denetlemek

*Ders: [YZ asistanıyla çalışmak, 3. bölüm: Bir eğitim döngüsünü denetlemek](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#ai-pair-programming-part-3-reviewing-a-training-loop)*

Bir katılımcı, bir YZ asistanından *dersin ağını test kesitlerinde değerlendirmesini* istemiştir. Aşağıdaki kod asistanın cevabıdır; kod çalışır ve bir doğruluk bildirir. Sayıya güvenmeden önce kodu dersteki eğitim döngüsü denetiminin altı sorusuyla okuyun: Ayırma hastaya göre mi yapılmış? Normalleştirmenin istatistikleri nereden gelmektedir? Model değerlendirme modunda mıdır ve gradyan kaydı kapalı mıdır? Çıktı katmanı kayıp fonksiyonuna uyuyor mu? Ölçüt sınıflara uygun mu?

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 9. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB03_derin_ogrenme_ve_evrisimli_aglar.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(9)

In [ ]:
# Bir YZ asistanının kodu: dersin ağını değerlendir
course_model = CmrCNN()
course_model.load_state_dict(model.state_dict())
X_all = torch.tensor((X - X.mean()) / X.std(), dtype=torch.float32)
probs = F.softmax(course_model(X_all[te]), dim=1)
accuracy = (probs.argmax(dim=1).numpy() == y[te]).mean()
print("Accuracy:", round(float(accuracy), 3))

**Gözlemle ve düşün.** Kodun üç sorunu ve bir eksiği vardır. Normalleştirmenin istatistikleri, test kesitleri dâhil 270 kesitin tamamından hesaplanmıştır. Yeni bir model eğitim modunda başlar; bu yüzden dropout birimleri rastgele kapatır, batch normalizasyonu da test kesitlerinin istatistiklerini kullanır. Autograd ise gereksiz yere her işlemin kaydını tutar. Doğruluk kesitler üzerinden sayılmıştır; oysa kliniğin her hasta için tek bir karara ihtiyacı vardır. Hücreyi iki kez çalıştırın: Doğruluk neden değişiyor?

**Alıştırma J.** `course_model` modelini doğru biçimde değerlendirin: Modeli değerlendirme moduna alın, eğitim kesitleriyle normalleştirilmiş `Xte` test kesitlerinin olasılıklarını `torch.no_grad()` içinde hesaplayın, olasılıkların her test hastasının kesitleri üzerinden ortalamasını alın ve 30 test hastasından kararı kendi sınıfıyla aynı olanların oranını hesaplayın. Sonucu üç ondalık basamağa yuvarlayın.

In [ ]:
veri = alistirma_tablosu("J")   # bu alıştırmanın promptundaki tablo (etkileşimli labın F bölümü)
cevap = None   # cevabınız
teach.check("Alıştırma J", cevap)

In [ ]:
#@title Örnek çözüm (bu hücreyi çalıştırın; açmayı denedikten sonraya bırakın)
_, _, true_j, pred_j = by_patient(course_model, Xte, te)
cevap = round(float((pred_j == true_j).mean()), 3)
teach.check("Alıştırma J", cevap)

Etkileşimli labdaki [prompt atölyesinde](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/lab.html#part-f) böyle bir değerlendirmenin promptu, prompt kartının yedi alanıyla yazılır. Denetimin bulduğu sorunları yakalayacak kontrol *Test* alanına yazılır: İki kez çalıştırılan aynı değerlendirme aynı sayıları vermelidir.

## 10. Kendi kaydınızla EchoNet-Dynamic

*Ders: [Mini proje: EchoNet-Dynamic veri setinden normal/anormal ekokardiyografi sınıflandırması](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/ders.html#echonet-dynamic-echocardiography-with-personal-registration)*

EchoNet-Dynamic, Stanford Medicine'e ait 10.030 apikal dört boşluk ekokardiyografi videosunu, ejeksiyon fraksiyonları ve diyastol sonu ile sistol sonunda uzmanlarca çizilmiş sınırlarla birlikte içerir. Her katılımcı Stanford AIMI'nin paylaşılan veri setlerine kaydolur, Araştırma Kullanım Sözleşmesi'ni kabul eder ve verileri kişisel olarak indirir. Aşağıdaki hücreler bu kopyayı okur: Kişisel indirme bağlantınızı ilk hücreye yapıştırın ya da zip dosyasını ders klasöründeki `data/echonet` klasörüne koyun. Kopya yoksa hücreler yalnızca bir mesaj yazdırır.

**Formüllerden koda.** `echonet.file_list(echo)` FileList.csv dosyasını okur; `echonet.frame_pairs(echo, split, n, size)` ise n videonun çizilmiş iki karesini 64 × 64 piksellik iki kanal olarak okur ve EF'si yüzde 50'nin altında olan videoları 1 olarak etiketler. 5. bölümdeki ağ burada iki çıktıyla kullanılır; ROC eğrisinin altındaki alan, ağın anormal videoları normal videolardan ne ölçüde önde sıraladığını ölçer.

**Buradan mı başlıyorsunuz?** Aşağıdaki telafi hücresini bir kez çalıştırın. Hücre, 10. bölümün ihtiyaç duyduğu önceki bölümlerin kodunu, çıktılarını göstermeden çalıştırır.

In [ ]:
#@title Telafi: Deftere buradan başlıyorsanız bu hücreyi bir kez çalıştırın
def _catch_up(upto, name="NB03_derin_ogrenme_ve_evrisimli_aglar.ipynb"):
    import contextlib, io, json, os, urllib.request
    import matplotlib.pyplot as plt
    if os.path.exists(name):
        with open(name, encoding="utf-8") as f:
            nb = json.load(f)
    else:
        url = ("https://raw.githubusercontent.com/utkukose/"
               "ai-cardiology-tkd-NB-lecture/main/tr/hafta-03/" + name)
        nb = json.loads(urllib.request.urlopen(url).read().decode("utf-8"))
    for cell in nb["cells"]:
        meta = cell.get("metadata", {})
        if cell["cell_type"] != "code" or "catch-up" not in meta.get("tags", []):
            continue
        if meta.get("section", 99) >= upto:
            continue
        with contextlib.redirect_stdout(io.StringIO()):
            exec("".join(cell["source"]), globals())
        plt.close("all")
    print("Telafi tamamlandı: Önceki bölümlerin kodu çalıştırıldı.")


_catch_up(10)

In [ ]:
# EchoNet-Dynamic kopyanız
ECHONET_LINK = ""   # kişisel indirme bağlantınızı, yalnızca veriler henüz bu bilgisayarda değilse buraya yapıştırın
try:
    echo = echonet.find()
except FileNotFoundError:
    echo = echonet.download(ECHONET_LINK) if ECHONET_LINK else None
print(echo if echo is not None else "EchoNet-Dynamic'in bir kopyası bulunamadı; sonraki hücreler bir kopya bekleyecek.")

In [ ]:
# Videolar, kümeleri ve ejeksiyon fraksiyonları
if echo is None:
    print("Önce EchoNet-Dynamic'e kaydolun (ders sayfasına bakın).")
else:
    file_table = echonet.file_list(echo)
    print(file_table["Split"].value_counts())
    fig, ax = plt.subplots(figsize=(6.4, 3.2))
    for split, colour in [("TRAIN", "#1F5F8B"), ("TEST", "#9B1B1B")]:
        ax.hist(file_table.loc[file_table["Split"] == split, "EF"], bins=40, alpha=0.6, density=True, color=colour, label=split)
    ax.axvline(50, color="#444444", ls="--", lw=1)
    ax.set_xlabel("EF (%)")
    ax.legend()
    plt.show()
    print("test kümesinde EF'si yüzde 50'nin altında olanların oranı",
          round(float((file_table.loc[file_table["Split"] == "TEST", "EF"] < 50).mean()), 3))

In [ ]:
# Dersin ağı 1.000 eğitim ve 400 test videosunda, birkaç dakika
if echo is None:
    print("Önce EchoNet-Dynamic'e kaydolun (ders sayfasına bakın).")
else:
    train_pairs = echonet.frame_pairs(echo, split="TRAIN", n=1000, size=64, seed=RANDOM_STATE)
    test_pairs = echonet.frame_pairs(echo, split="TEST", n=400, size=64, seed=RANDOM_STATE)
    e_mean, e_sd = float(train_pairs["images"].mean()), float(train_pairs["images"].std())
    Xe_tr = torch.tensor((train_pairs["images"] - e_mean) / e_sd, dtype=torch.float32)
    Xe_te = torch.tensor((test_pairs["images"] - e_mean) / e_sd, dtype=torch.float32)
    ye_tr = torch.tensor(train_pairs["abnormal"], dtype=torch.long)
    torch.manual_seed(RANDOM_STATE)
    echo_model = CmrCNN(n_classes=2)
    optimizer = torch.optim.Adam(echo_model.parameters(), lr=0.003)
    for epoch in range(15):
        echo_model.train()
        order = torch.randperm(len(ye_tr))
        for start in range(0, len(ye_tr), 32):
            j = order[start:start + 32]
            loss = F.cross_entropy(echo_model(Xe_tr[j]), ye_tr[j])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    echo_model.eval()
    with torch.no_grad():
        p_abnormal = F.softmax(echo_model(Xe_te), dim=1)[:, 1].numpy()
    print("EF'nin yüzde 50'nin altında olması için AUC:", round(roc_auc_score(test_pairs["abnormal"], p_abnormal), 3))

**Python notu.** `if echo is None:`, hücrenin kopya olmadan da çalışmasını ve hata yerine bir mesaj yazdırmasını sağlar. `echonet.frame_pairs` örneklemdeki her videoyu açar, çizilmiş iki kareyi alır ve 64 × 64 piksele küçültür; 1.400 videoyu okumak bir iki dakika sürer. Ağ ve döngü 5. bölümdekilerdir; yalnızca çıktı sayısı değişir. Videolar Colab oturumunuzda ya da bilgisayarınızda kalır.

## Alanlara göre uygulama görevleri

| Alan | Görev |
|---|---|
| Kalp yetersizliği | Geçirilmiş miyokard enfarktüsü grubunu (MINF) `acdc.pairs(("NOR", "DCM", "HCM", "MINF"))` ile ekleyin ve `CmrCNN(n_classes=4)` modelini eğitin. Ağ hangi grupları karıştırıyor ve DCM ile büyük bir enfarktüs ortadaki kesitlerde neden birbirine benzeyebilir? |
| Girişimsel kardiyoloji | Ağı 15, 30 ve 45 eğitim hastasıyla eğitin ve 30 test hastasındaki doğruluğu eğitim hastası sayısına karşı çizin. Yeni bir laboratuvar, böyle bir ağı eğitmeye değecek duruma gelmeden önce kaç inceleme toplamalıdır? |
| Elektrofizyoloji | Her test hastasının kalp döngüsü başına kare sayısını (NbFrame) `acdc.read_info` ile okuyun ve ağın hatalarını 20'den az ve 20'den çok kare içeren incelemelerde karşılaştırın. Kalp hızı ve EKG tetiklemesi sine görüntüler için neden önemlidir? |
| Kardiyovasküler görüntüleme | `acdc.pairs(GROUPS, n_slices=5)` ile hasta başına ortadaki beş kesiti kullanın ve ağı yeniden eğitin. Ağ şimdi 134 numaralı hastayı tanıyor mu? Kesit dizisinin tamamını, tabandan apekse okuyan bir ağın nelere ihtiyacı olurdu? |
| Koruyucu kardiyoloji | HCM'li hastaların yakınlarının taranmasında hastalık kaçırılmamalıdır. Doğrulama hastalarında HCM'li bütün hastaları yakalayan eşiği, yani HCM'li hastaların aldığı en düşük HCM olasılığını seçin; bu eşiği test hastalarına uygulayın ve HCM için duyarlılığı ve kesinliği bildirin. |
| Kardiyo-onkoloji | Ardışık ejeksiyon fraksiyonu ölçümleri kardiyo-onkolojide kararlara yol gösterir. Her test hastasının konturlardan elde edilen EF'sini, ağın verdiği DCM olasılığına karşı çizin. Ağ, EF'nin yüzde 60'tan 50'ye düşüşünü ölçüm kadar erken fark edebilir miydi? |
| Yoğun bakım | 90 test kesitinin tahmin süresini kendi bilgisayarınızda ölçün ve saniyedeki kesit sayısını hesaplayın. Ağ, her biri 30 kare içeren 10 kesitlik bütün bir kesit dizisi için ne kadar süreye ihtiyaç duyardı? |

## Defterden sonra

[Etkileşimli labı](https://utkukose.github.io/ai-cardiology-tkd-NB-lecture/tr/hafta-03/lab.html) açın ve *Kendini sına* sekmesindeki öz değerlendirmeyi yapın. Ardından *Düşün ve dışa aktar* sekmesinde yansıtma metnini yazın ve öğrenme günlüğünü dışa aktarın. [Hafta sayfasında](README.md) anlatılan 3. haftanın haftalık projesi, 5. bölümdeki ağı 30 test hastasında ACDC kurallarıyla karşılaştırır.

<div align="center">

---

**Prof. Dr. Utku Köse**<br>
Profesör, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi, Isparta, Türkiye<br>
Kurucu Müdür, Yapay Zekâ Uygulama ve Araştırma Merkezi (YAZEM), Süleyman Demirel Üniversitesi<br>
Bilgisayar Bilimleri Anabilim Dalı Başkanı, Bilgisayar Mühendisliği Bölümü, Süleyman Demirel Üniversitesi<br>
Diğer kurumlar: University of North Dakota (ABD), Universidad Panamericana (Meksiko, Meksika), Vel Tech University (Chennai, Hindistan)<br>
IEEE Kıdemli Üyesi, ACM Profesyonel Üyesi

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) · [utkukose.com](https://www.utkukose.com) · [github.com/utkukose](https://github.com/utkukose)

*Python ile Yapay Zekâ Eğitimi: Kardiyologlar için Uygulamalı Eğitim Programı, Türk Kardiyoloji Derneği. İlk sürüm: Kasım 2026. İçerik CC BY 4.0, kod MIT.*

</div>